# Masked Concept Subset Attention MC2 v2: record of experiments

This notebook is standalone. Every number the cache paper takes from the masked concept pilot is produced by a cell in this notebook or carried into this record from the superseded MC2 notebook with its contract, fingerprint and result file. Nothing here refers to another notebook for a result.

## Claims supported here

| Claim in the paper | Cell | Contract or fingerprint | Result file | Result |
|---|---|---|---|---|
| A baseline that reads context exists on the cleaned task; D4 is the update rule | MC3ae, MC3af | `f1f3d18f90b0b97f` | `mc3ae_curve.pkl.lz4`, `mc3_decile_edges.pkl.lz4` | W0 under D4, epoch 2: 0.1665 against concept median 0.2242 and zero 0.2346; D4 minus registered −0.0523 [−0.0564, −0.0486]; shuffle probe 0.13 against 0.0005 for the registered rule |
| Subset supervision through the cache (W) improves the baseline, one seed | MC3ag | `e9588ca98bddc1ea` | `mc3ag_curve.pkl.lz4` | W epoch 2 minus W0 epoch 2: −0.00728 [−0.00842, −0.00628], 4.4% lower pooled loss; W 6.24 h against W0 0.73 h |
| Exhaustively evaluated subset supervision over a ranked context (T) improves its baseline, one seed | MC3ah | `972f58d9b99de2d0` | `mc3ah_curve.pkl.lz4` | T epoch 2 minus T0 epoch 2: −0.00174 [−0.00241, −0.00108], 1.1% lower pooled loss on the ranked context evaluation; T 4.06 h against T0 0.84 h; bins 0 to 4 (55.5% of targets) favor T0 |
| The pointer jumping walk reproduces the level walk | MC2k (carried); MC3ag0 | float64 tolerance 1e-9; init `88071b7a2e52af9f`, plans `c9ec2fc1ce2e0860`, `8aef1402fd7de8d8` | printed in cells | MC2k: 20 of 20 checks passed, loss, predictions, gradients and masks identical to 5e-15, W walks of 46 to 203 levels in 6 to 8 rounds, T walks of 12 levels in 4 rounds; W single batches 4.9 to 7.7 times faster, full batches 1.7 to 2.3 times, T full batches 0.4 to 0.5 times (T stays on the level walk). MC3ag0: through the full W objective, loss difference 8.9e-10, gradient max relative difference 1.6e-8, jump 0.43 s against levels 1.75 s on a 37 target group |
| Scoped direct enumeration is competitive with the cache at pilot scale; both paths train the same objective | MC3ai, MC3ai1 | sources `f53957e0033d0c72`, `647b44ba68fe9dd4`, `e8c9321790cbddd3`, `f7b8e10a079077d5`, `0be0f3440e96db76` | `mc3ai_results.pkl.lz4`, `mc3ai_output.txt` | per tensor gradient equivalence at every chunk size; forward and backward hours per epoch: W cache 2.54 against direct 2.07 (ratio 0.81), T cache 1.86 against direct 2.39 (ratio 1.29); direct peak 0.52 GiB against cache 2.65 GiB |

Primary endpoints were registered before each run: epoch 2 against epoch 2, pooled mean Huber, paired by filing, filing clustered bootstrap 95% interval (2,000 resamples, seed (367, 91)). Intervals measure held out filing sampling uncertainty for the trained models and do not include training seed variability. Subgroup intervals are nominal and descriptive. All results are one seed (367).

## Data (built in MC2a, MC2b, MC3ab, MC3ac, MC3ad; carried here, not rerun)

- Source: `GaapFeaturesML` window 2019 to 2025 (3,364,836 filings, 380,176 concepts, 22.5 million facts; name store fingerprint `56f270b8e5f2c1b1`), restricted to GAAP percentage change concepts with the QoQ, YoY and FYED to DPED suffixes. A pct change exists only for an exactly matched period pair, so the window is part of each concept's identity.
- Pilot draw (MC2a): 10,000 training filings drawn from 137,093 eligible filings dated 2019 to 2024 Q2, and 1,000 held out filings drawn from 12,002 eligible filings dated 2024 Q3 and Q4. The split is out of time. Filings carry 4 to 501 facts (median 123 training, 139 held out); 110,070 distinct concepts across the 11,000 filings; 37% of held out concepts never occur in training. Uncleaned facts fingerprint `a62fb1b9b0a3644c`.
- D5 cleaning (MC3ab, MC3ac), the production artifact rule from `gaap_feature_rankings_reduce.py`: entries with |raw pct change| > 100 (fraction units, 10,000%) dropped; filings left with fewer than 2 facts dropped. Removed 51,838 of 1,286,742 training facts (4.03%, in 74.5% of filings) and 5,231 of 141,679 held out facts (3.69%); 3 training and 2 held out filings dropped; 102,395 training concepts became 95,429 (6,966 had every occurrence above the threshold). Before cleaning, the artifacts carried 75.2% of the held out loss at prediction zero from 3.69% of targets. Every removed occurrence is recorded in `mc3_pilot_data_d5_removed.pkl.lz4` (57,069 rows).
- Cleaned pilot data `mc3_pilot_data_d5.pkl.lz4`, facts fingerprint `1d6ee181e9fd1807`: 1,234,902 training occurrences, 126,767 column microbatches per epoch (61,105 single, 29,405 small, 4,794 medium, 31,463 full); 136,446 held out targets; largest remaining |raw| 100 (|signed log| 4.615).
- Contexts `mc3_contexts_d5.pkl.lz4`, fingerprint `bb82e1d833c2983b` (MC3ad, the MC2b procedure on cleaned data): for every fact, the top 12 (and 16) other facts in its filing ranked by |Pearson correlation| of signed log values over training filings holding both concepts, computed only for concept pairs sharing at least 50 training filings, with the filing's most common concepts as fallback; 2,281 concepts with 50 or more cleaned training filings; 233,035 scored pairs; at N 12, 99.9% of targets have a full context and 82% of chosen facts are scored (mean |r| 0.34). W0 and W attend to the whole filing; T0 and T to these 12 ranked facts.
- Held out references on the cleaned targets: zero predictor 0.2346; per concept training median (global training median for unseen concepts) 0.2242.
- Report bins: ten bins on |signed log y| at training decile cut points, right edge inclusive (`mc3_decile_edges.pkl.lz4`; value boundaries 0.030, 0.078, 0.150, 0.257, 0.422, 0.667, 0.998, 1.168, 2.75 in fraction units); the magnitude bins 0 to 1, 1 to 3, 3 to 10 retained for continuity; concept groups by distinct training filings per concept (seen_common 50 or more, seen_rare 1 to 49, unseen 0).

## Deviation register

| ID | Deviation from the specification | Status |
|---|---|---|
| D1 | W subset budget M 2,048 per filing draw (specification: up to 16,384); W run length n (run_factor 1; specification 10n); W on the pointer jumping walk (certified equal to the level walk in MC2k); T on the level walk. No narrowing of features or targets; the Horvitz Thompson estimator unchanged. Basis: n M2048 costs about the same as 10n M4096 (28.9 h against 30.4 h for 8 W epochs) with 6.4% against 29.4% zero usable draws (MC2m, MC2n). Not established: cost or coverage at M 16,384 | approved 10/17 |
| D3 | Column batch ordering interleaved: the same one concept microbatches, every training occurrence targeted once per epoch, in a seeded random order (367, epoch, 131) instead of each concept's batches consecutive. Basis: MC3n, MC3p, MC3r; under contiguous order the encoder's keys aligned by the first measurement and the model did not fit; interleaved order learned with context sensitivity recovering | approved for MC3s; used by every later arm |
| D4 | Accumulated optimizer updates: per target losses summed over successive microbatches, gradients divided by the accumulated target count, one AdamW step at 32 or more targets or at epoch end; learning rate by target exposure (warmup over the first 10% of one epoch's targets, cosine over 8 epochs' targets, evaluated at the group end). Gradient composition, Adam history, update count and cumulative weight decay all change; which produces the gain is not claimed. Basis: MC3x against MC3w, then MC3z and MC3aa at full scale on uncleaned data | adopted 10/17; confirmed on cleaned data in MC3ae (D4 minus registered −0.0523 [−0.0564, −0.0486]) |
| D5 | Artifact exclusion by the production rule and the drop of filings with fewer than 2 facts | built 10/17 (MC3ac); contexts rebuilt (MC3ad) |
| S1 | W sampler supervision per draw: in MC3ag, 5.4% of sampled targets received no usable subset in a microbatch (that target's loss in the microbatch is the full context contribution as weighted in `mc_target_loss`); half of drawn subsets usable; P median 1,980 against requested 2,048, minimum 3 | open |
| E1 | Verdict measure: pooled mean Huber stays primary; bins, bands and groups expose tradeoffs and do not gate | ruled 10/18 |

## Limits stated in the paper

One seed. Filing intervals conditional on the trained models. Equal updates and equal data exposure, not equal compute (W 8.6 times, T 4.8 times the baseline's training time). W and T evaluate different context policies with different subset terms and weightings, so their effect sizes cannot be compared to isolate any one factor. Small magnitude targets regress under T (bins 0 to 4) and show mixed results under W (bins 0 and 3). The direct enumeration baseline is target by target and chunked; a batched direct implementation could be faster. The cache itself is not shown to improve accuracy; it computes the subset objectives. The held out set was used repeatedly during development.

## Environment

RTX 2080 SUPER (8 GiB), torch 2.5.1+cu121, CUDA 12.1, Python 3.12.7, Windows. Module hashes (inspect based, as used in every contract): `set_attention_cache` `f53957e0033d0c72`, `set_attention_model` `647b44ba68fe9dd4`, `masked_concept_model` `e8c9321790cbddd3`, `mc_pilot_data` `f7b8e10a079077d5`, `mc_batched` `0be0f3440e96db76`.

## Result files (repo `results\mc3`, sha256 first 16)

`mc3ae_curve.pkl.lz4` 03A5D9A7381DB307; `mc3ag_curve.pkl.lz4` 8EFAA8CF97DAFA00; `mc3ah_curve.pkl.lz4` 365BEF2D4EE47D6E; `mc3_decile_edges.pkl.lz4` F6270354A9B66881; `mc3ai_results.pkl.lz4` F91681F8D22D6918; `mc3ai_output.txt` 2E04643BA4D57E5D. Checkpoints (`mc3ae_*.pt`, `mc3ag_W_*.pt`, `mc3ah_*.pt`) remain under `E:\apriori-ml\mc3` and are named in each cell's output.i-ml\mc3` and are named in each cell's output.diction lift.

## MC3ae. Matched comparison on the cleaned data: W0 under the registered update rule against W0 under D4

**In plain terms.** Two baselines train on the D5 cleaned data from the same initial weights, over the same epoch specific microbatch sequences (D3 interleaved order), for two epochs each. One updates after every microbatch with the registered learning rate schedule; the other accumulates at least 32 targets per update with D4's target exposure schedule. Both are scored on the same cleaned held out targets at every 10% and compared target by target at each epoch end with paired, filing clustered intervals. This is the first comparison on the cleaned task, and the only one that can carry a claim about D4 there. Nothing from the uncleaned runs is reused.

**Printed: header:** the D5 fingerprints, the schedule for each rule, batch counts, and the zero and concept median references on the cleaned held out targets.

**Printed at each 10% boundary and epoch end, per rule:** `rule`, `epoch`, `pct`, `opt_steps`, `lr`, `heldout`, `0 to 1`, `seen_common`, `d_pred_shuffle`, `hours`.

**Printed at the end**

- Per rule and epoch: `heldout`, bands (`0 to 1`, `1 to 3`, `3 to 10`; the 10+ band is empty on cleaned data), groups (`seen_common`, `seen_rare`, `unseen`), `d_pred_shuffle`, `opt_steps`, `hours`, `grad_max_windows`, `upd_med_windows`, `peak_gib`.
- Paired comparison per epoch: `d_mean` (D4 minus registered per held out target; negative favors D4), `ci_lo`, `ci_hi` (95%, bootstrap over filings), overall, by band and by group.

Files under `E:\apriori-ml\mc3`: `mc3ae_{rule}_last.pt`, `mc3ae_{rule}_epoch{1,2}.pt`, `mc3ae_curve.pkl.lz4` (rows, per target losses, contract). Resumable per rule.

In [1]:
# MC3ae. Cleaned data (D5): W0 under the registered update rule vs W0 under D4; two epochs each; paired comparison.
import os, sys, time, math, re, hashlib, json, inspect
from dataclasses import asdict
os.environ["ENV"] = "prod"
for p_ in (r"E:\apriori-ml", r"C:\Users\jakem\Documents\Belk\Sec\Bloomberg Pipeline v1\BB_Pipeline_v15"):
    if p_ not in sys.path:
        sys.path.insert(0, p_)
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import importlib
import set_attention_cache, set_attention_model
import masked_concept_model as mcm
import mc_pilot_data as mpd
import mc_batched as mcb
importlib.reload(mcm); importlib.reload(mpd); importlib.reload(mcb)
from set_attention_model import signed_log
from apriori.serialization import load_from_pickle, save_to_pickle

HARNESS_VERSION = "mc3ae-1"
OUT_DIR = r"E:\apriori-ml\mc3"
PILOT, CTXF = os.path.join(OUT_DIR, "mc3_pilot_data_d5.pkl.lz4"), os.path.join(OUT_DIR, "mc3_contexts_d5.pkl.lz4")
CURVE = os.path.join(OUT_DIR, "mc3ae_curve.pkl.lz4")
FACTS_FP = "1d6ee181e9fd1807"
KIND, RULES, EPOCHS_RUN, EPOCHS_REGISTERED = "W0", ["registered", "D4"], 2, 8
M_W, RUN_FACTOR, N_T, BATCH, MASK_RATE, ORDER_SEED_TAG = 2048, 1, 12, 32, 0.15, 131
LR, WD, WARMUP_FRAC, MIN_TARGETS = 3e-4, 0.01, 0.10, 32
WORKERS, SEED, EVAL_CHUNK, N_PROBE, LOADER_TIMEOUT, MAX_STALLS, BOOT = 16, 367, 1024, 20, 300, 20, 2000
BANDS = [(0, 1, "0 to 1"), (1, 3, "1 to 3"), (3, 10, "3 to 10"), (10, np.inf, "10+")]
dev, f32 = "cuda", torch.float32

def sha16(b):
    return hashlib.sha256(b).hexdigest()[:16]

def huber(pred, y):
    r = np.abs(pred - y)
    return np.where(r <= 1.0, 0.5 * r * r, r - 0.5)

# ----- data (D5) -----
pdat = load_from_pickle(PILOT)
assert pdat["fingerprints"]["facts"] == FACTS_FP and "D5" in pdat.get("deviation", {})
cdat = load_from_pickle(CTXF)
assert cdat["pilot_facts_fp"] == FACTS_FP
indptr, indices, data = pdat["indptr"], pdat["indices"], pdat["data"]
split = np.asarray(pdat["split"])
tr_idx, ho_idx = np.flatnonzero(split == "train"), np.flatnonzero(split == "holdout")
len_a = np.diff(indptr)
NT = torch.from_numpy(pdat["name_table"].astype(np.float32)).to(dev)
ctx12 = cdat["ctx"][N_T]
doc_freq = cdat["doc_freq_train"]
sl = signed_log(torch.as_tensor(np.asarray(data, dtype=np.float32))).numpy().astype(np.float64)
fact_row = np.repeat(np.arange(len(split)), len_a)
fact_local = np.arange(len(indices)) - indptr[fact_row]
tr_fact = np.isin(fact_row, tr_idx)
occ_filing, occ_local, occ_concept = fact_row[tr_fact], fact_local[tr_fact], indices[tr_fact]
N_OCC = len(occ_filing)

def epoch_keys(e):
    plan = mpd.column_epoch_plan(occ_concept, e, SEED, BATCH)
    order = np.random.default_rng((SEED, e, ORDER_SEED_TAG)).permutation(len(plan))
    plan = [plan[i] for i in order]
    allocc = np.concatenate([occ for _, occ in plan])
    assert len(allocc) == N_OCC and np.array_equal(np.sort(allocc), np.arange(N_OCC)), "plan coverage"
    return [[(int(occ_filing[o]), e, 1, (int(occ_local[o]),)) for o in occ] for _, occ in plan]

UPDATES_PER_EPOCH = len(epoch_keys(0))
BOUNDS = [int(round(UPDATES_PER_EPOCH * k / 10)) for k in range(1, 11)]
WARM_U, TOTAL_U = int(round(WARMUP_FRAC * UPDATES_PER_EPOCH)), UPDATES_PER_EPOCH * EPOCHS_REGISTERED
WARM_T, TOTAL_T = WARMUP_FRAC * N_OCC, N_OCC * EPOCHS_REGISTERED

def lr_registered(g):
    return LR * (g + 1) / WARM_U if g < WARM_U else LR * 0.5 * (1 + math.cos(math.pi * min(g - WARM_U, TOTAL_U - WARM_U) / max(TOTAL_U - WARM_U, 1)))

def lr_d4(t):
    return LR * t / WARM_T if t < WARM_T else LR * 0.5 * (1 + math.cos(math.pi * min(t - WARM_T, TOTAL_T - WARM_T) / max(TOTAL_T - WARM_T, 1)))

# ----- held out targets, bands, groups, probe -----
ho_rows = np.concatenate([np.arange(int(indptr[i]), int(indptr[i + 1])) for i in ho_idx])
y_ho, c_ho = sl[ho_rows], indices[ho_rows].astype(np.int64)
fid = fact_row[ho_rows]
ay = np.abs(y_ho)
bands = {nm: (ay >= lo) & (ay < hi) for lo, hi, nm in BANDS}
df_ho = doc_freq[c_ho]
groups = {"seen_common": df_ho >= 50, "seen_rare": (df_ho >= 1) & (df_ho < 50), "unseen": df_ho == 0}
tr_c, tr_y = indices[tr_fact].astype(np.int64), sl[tr_fact]
med = pd.Series(tr_y).groupby(tr_c).median()
c_med = np.full(int(indices.max()) + 1, float(np.median(tr_y))); c_med[med.index.values] = med.values
probe_f = np.sort(np.random.default_rng((SEED, 181)).choice(ho_idx[len_a[ho_idx] <= 1024], N_PROBE, replace=False))
probe = []
for i in probe_f:
    lo, hi = int(indptr[i]), int(indptr[i + 1])
    raw = torch.as_tensor(np.asarray(data[lo:hi], dtype=np.float32), device=dev)
    perm = torch.as_tensor(np.random.default_rng((SEED, 171, int(i))).permutation(hi - lo), device=dev)
    probe.append((NT[torch.as_tensor(indices[lo:hi].astype(np.int64), device=dev)], raw, raw[perm]))

PARAMS = {"arm": KIND, "data": "D5", "order": "interleaved", "order_seed_tag": ORDER_SEED_TAG, "M_W": M_W, "run_factor": RUN_FACTOR,
          "batch": BATCH, "epochs_registered": EPOCHS_REGISTERED, "epochs_run": EPOCHS_RUN, "lr": LR, "wd": WD,
          "registered": {"warmup_updates": WARM_U, "total_updates": TOTAL_U},
          "D4": {"min_targets_per_step": MIN_TARGETS, "warmup_targets": WARM_T, "total_targets": TOTAL_T, "lr_by": "target exposure, group end"},
          "seed": SEED, "eval": "cleaned heldout, all facts, leave one out, full context"}
CONTRACT = sha16(json.dumps({"sources": {m.__name__: sha16(inspect.getsource(m).encode()) for m in (set_attention_cache, set_attention_model, mcm, mpd, mcb)},
                             "facts_fp": FACTS_FP, "contexts_fp": cdat["fingerprint"], "split": sha16(split.astype("U").tobytes()),
                             "mc_config": asdict(mcm.MCConfig()), "params": PARAMS, "harness_version": HARNESS_VERSION},
                            sort_keys=True, default=str).encode())
ds = mpd.PilotDataset(indptr, indices, data, pdat["file_names"], ctx12, KIND, M=M_W, run_factor=RUN_FACTOR, seed=SEED, rate=MASK_RATE)

def new_model():
    torch.manual_seed(SEED)
    return mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32)

@torch.no_grad()
def predict_filing(model, nv, vals):
    n = nv.shape[0]
    preds = []
    for s in range(0, n, EVAL_CHUNK):
        tg = torch.arange(s, min(n, s + EVAL_CHUNK), device=dev)
        c = torch.arange(len(tg), device=dev)
        z, V, e = model.scores(nv, vals, nv[tg])
        member = torch.ones(len(tg), n, dtype=torch.bool, device=dev)
        member[c, tg] = False
        fp, fL, fk, fu = mcm.mc_masked_attention(z, V, member, c)
        assert bool(fu.all())
        preds.append(model.predict(fp, fL, fk, e))
    return torch.cat(preds).double().cpu().numpy()

@torch.no_grad()
def evaluate(model):
    t0 = time.perf_counter()
    rc, rg = torch.get_rng_state(), torch.cuda.get_rng_state_all()
    model.eval()
    out = []
    for i in ho_idx:
        lo, hi = int(indptr[i]), int(indptr[i + 1])
        out.append(predict_filing(model, NT[torch.as_tensor(indices[lo:hi].astype(np.int64), device=dev)],
                                  torch.as_tensor(np.asarray(data[lo:hi], dtype=np.float32), device=dev)))
    pred = np.concatenate(out)
    assert np.isfinite(pred).all()
    dp, pb = [], []
    for nv, raw, shuf in probe:
        p0, p1 = predict_filing(model, nv, raw), predict_filing(model, nv, shuf)
        dp.append(np.abs(p1 - p0)); pb.append(p0)
    model.train()
    torch.set_rng_state(rc); torch.cuda.set_rng_state_all(rg)
    l = huber(pred, y_ho)
    ev = {"heldout": float(l.mean()), **{k: float(l[m].mean()) if m.any() else float("nan") for k, m in bands.items()},
          **{k: float(l[m].mean()) for k, m in groups.items()},
          "d_pred_shuffle": float(np.median(np.concatenate(dp))), "pred_spread": float(np.std(np.concatenate(pb)))}
    return l, ev, time.perf_counter() - t0

def save_ckpt(path, model, opt, st):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "st": st, "contract": CONTRACT,
                "rng_cpu": torch.get_rng_state(), "rng_cuda": torch.cuda.get_rng_state_all()}, path + ".tmp")
    os.replace(path + ".tmp", path)

def cluster_ci(d, f):
    uniq, inv = np.unique(f, return_inverse=True)
    s, c = np.bincount(inv, weights=d), np.bincount(inv)
    idx = np.random.default_rng((SEED, 91)).integers(0, len(uniq), size=(BOOT, len(uniq)))
    est = s[idx].sum(1) / c[idx].sum(1)
    return float(d.mean()), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

sizes = np.array([len(k) for k in epoch_keys(0)])
print(f"MC3ae contract {CONTRACT} | D5 facts {FACTS_FP} | contexts {cdat['fingerprint']}")
print(f"epoch: {UPDATES_PER_EPOCH:,} microbatches, {N_OCC:,} targets; single {int((sizes == 1).sum()):,}, small {int(((sizes >= 2) & (sizes <= 8)).sum()):,}, "
      f"medium {int(((sizes >= 9) & (sizes <= 31)).sum()):,}, full {int((sizes == 32).sum()):,}")
print(f"registered rule: update per microbatch, warmup {WARM_U:,} updates, cosine to {TOTAL_U:,} | D4: step at >= {MIN_TARGETS} targets, "
      f"warmup {WARM_T:,.0f} targets, cosine to {TOTAL_T:,}")
lz, lm = huber(np.zeros_like(y_ho), y_ho), huber(c_med[c_ho], y_ho)
print(f"cleaned held out targets {len(y_ho):,}: zero {lz.mean():.4f} | concept_median {lm.mean():.4f} | "
      + " | ".join(f"{k} {int(m.sum()):,}" for k, m in bands.items()))

curve = load_from_pickle(CURVE) if os.path.exists(CURVE) else {}
if curve:
    assert curve.get("contract") == CONTRACT, f"{CURVE} was written under a different contract; delete it and mc3ae_*.pt to start over"
else:
    curve = {"contract": CONTRACT, "params": PARAMS, "y_ho": y_ho, "c_ho": c_ho, "fid": fid, "rows": [], "epochs": {r: {} for r in RULES}}

for rule in RULES:
    if all((ep in curve["epochs"][rule]) for ep in range(1, EPOCHS_RUN + 1)):
        print(f"[{rule}] complete; skipping")
        continue
    model = new_model()
    params = list(model.parameters())
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
    last = os.path.join(OUT_DIR, f"mc3ae_{rule}_last.pt")
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)
        assert ck["contract"] == CONTRACT
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); st = ck["st"]
        torch.set_rng_state(ck["rng_cpu"]); torch.cuda.set_rng_state_all(ck["rng_cuda"])
        print(f"[{rule}] resumed: epoch index {st['e']}, microbatch {st['k']:,}")
    else:
        _, ev0, t_ev = evaluate(model)
        st = {"e": 0, "k": 0, "g": 0, "t": 0, "steps": 0, "train_s": 0.0, "eval_s": t_ev, "gwin": [], "gmax": 0.0, "uwin": [], "ups": [],
              "tps": [], "peak": 0.0, "stalls": [], "pending_ckpt": False}
        curve["rows"].append({"rule": rule, "epoch": 0, "pct": 0, "opt_steps": 0, "lr": 0.0, **ev0, "hours": t_ev / 3600})
        print(f"[{rule}] init: heldout {ev0['heldout']:.4f}")
    model.train()
    torch.cuda.reset_peak_memory_stats()
    acc_n = 0
    opt.zero_grad(set_to_none=True)

    def do_step(count, lr):
        if count > 1 or rule == "D4":
            for p in params:
                if p.grad is not None:
                    p.grad.div_(count)
        grads = [p.grad for p in params if p.grad is not None]
        if not all(bool(torch.isfinite(q).all()) for q in grads):
            raise RuntimeError(f"[{rule}] nonfinite gradient at step {st['steps']}; stopped before the update")
        st["gmax"] = max(st["gmax"], float(torch.sqrt(sum((q.double() ** 2).sum() for q in grads))))
        for gr in opt.param_groups:
            gr["lr"] = lr
        before = [p.detach().clone() for p in params]
        opt.step()
        st["ups"].append(float(torch.sqrt(sum(((p.detach() - b).double() ** 2).sum() for p, b in zip(params, before)))))
        opt.zero_grad(set_to_none=True)
        st["steps"] += 1; st["tps"].append(count)

    for e in range(st["e"], EPOCHS_RUN):
        if (e + 1) in curve["epochs"][rule]:
            continue
        keys = epoch_keys(e)
        epoch_done = False
        while not epoch_done:
            loader = DataLoader(ds, batch_sampler=keys[st["k"]:], collate_fn=mpd.collate_list, num_workers=WORKERS, prefetch_factor=2, timeout=LOADER_TIMEOUT)
            t_seg = time.perf_counter()
            try:
                for items in loader:
                    k = st["k"]
                    assert [it["i"] for it in items] == [key[0] for key in keys[k]], "loader out of plan order"
                    n = len(items)
                    loss, _, _, _ = mcb.batch_loss(model, KIND, items, NT, dev)
                    if not bool(torch.isfinite(loss)):
                        raise RuntimeError(f"[{rule}] nonfinite loss at epoch {e + 1} microbatch {k}")
                    st["k"] += 1; st["t"] += n
                    end = st["k"] == UPDATES_PER_EPOCH
                    if rule == "registered":
                        loss.backward()
                        do_step(1, lr_registered(st["g"])); st["g"] += 1
                    else:
                        (loss * n).backward()
                        acc_n += n
                        if acc_n >= MIN_TARGETS or end:
                            do_step(acc_n, lr_d4(st["t"])); acc_n = 0
                            if st["pending_ckpt"]:
                                st["train_s"] += time.perf_counter() - t_seg; t_seg = time.perf_counter()
                                save_ckpt(last, model, opt, st); st["pending_ckpt"] = False
                    if st["k"] in BOUNDS:
                        st["train_s"] += time.perf_counter() - t_seg
                        ups = np.array(st["ups"]) if st["ups"] else np.array([np.nan])
                        st["uwin"].append(round(float(np.median(ups)), 4)); st["ups"] = []
                        st["gwin"].append(round(st["gmax"], 2)); st["gmax"] = 0.0
                        st["peak"] = max(st["peak"], torch.cuda.max_memory_allocated() / 2**30)
                        pct = BOUNDS.index(st["k"]) * 10 + 10
                        l_new, ev, t_ev = evaluate(model)
                        st["eval_s"] += t_ev
                        lr_now = lr_registered(st["g"]) if rule == "registered" else lr_d4(st["t"])
                        row = {"rule": rule, "epoch": e + 1, "pct": pct, "opt_steps": st["steps"], "lr": lr_now, **ev, "hours": (st["train_s"] + st["eval_s"]) / 3600}
                        curve["rows"].append(row)
                        print(f"[{rule}] epoch {e + 1} {pct}%: steps {st['steps']:,} | heldout {ev['heldout']:.4f} | 0 to 1 {ev['0 to 1']:.4f} | "
                              f"seen_common {ev['seen_common']:.4f} | d_pred_shuffle {ev['d_pred_shuffle']:.4g} | lr {lr_now:.2e} | {row['hours']:.2f} h")
                        if end:
                            tps = np.array(st["tps"])
                            curve["epochs"][rule][e + 1] = {"losses": l_new, "row": row, "grad_max_windows": st["gwin"], "upd_med_windows": st["uwin"],
                                                            "peak_gib": st["peak"], "targets_per_step": (float(np.median(tps)), float(np.percentile(tps, 90)), int(tps.max()))}
                            save_ckpt(os.path.join(OUT_DIR, f"mc3ae_{rule}_epoch{e + 1}.pt"), model, opt, st)
                            st.update({"e": e + 1, "k": 0, "gwin": [], "uwin": [], "tps": [], "pending_ckpt": False})
                            save_ckpt(last, model, opt, st)
                            epoch_done = True
                        elif rule == "registered" or acc_n == 0:
                            save_ckpt(last, model, opt, st)
                        else:
                            st["pending_ckpt"] = True
                        save_to_pickle(curve, CURVE, lz4_compression=True)
                        t_seg = time.perf_counter()
                        if epoch_done:
                            break
            except RuntimeError as ex:
                if "DataLoader timed out" not in str(ex):
                    raise
                st["stalls"].append((e, st["k"]))
                print(f"[{rule}] loader stalled at epoch {e + 1} microbatch {st['k']:,}; rebuilding (stall {len(st['stalls'])} of at most {MAX_STALLS})")
                if len(st["stalls"]) >= MAX_STALLS:
                    raise RuntimeError(f"[{rule}] {MAX_STALLS} loader stalls; stopping") from None
                st["train_s"] += time.perf_counter() - t_seg
            finally:
                del loader
    del model, opt, params
    torch.cuda.empty_cache()

pd.set_option("display.float_format", lambda v: f"{v:.4g}")
rows = []
for rule in RULES:
    for ep in sorted(curve["epochs"][rule]):
        r, info = curve["epochs"][rule][ep]["row"], curve["epochs"][rule][ep]
        rows.append({"rule": rule, "epoch": ep, **{k: r[k] for k in ("heldout", "0 to 1", "1 to 3", "3 to 10", "seen_common", "seen_rare", "unseen",
                                                                      "d_pred_shuffle", "opt_steps", "hours")},
                     "grad_max_windows": info["grad_max_windows"], "upd_med_windows": info["upd_med_windows"], "peak_gib": info["peak_gib"]})
print("\nMC3ae W0 ON CLEANED DATA BY RULE AND EPOCH")
print(pd.DataFrame(rows).to_string(index=False))
print(f"references: zero {lz.mean():.4f} | concept_median {lm.mean():.4f}")
comp = []
for ep in range(1, EPOCHS_RUN + 1):
    if ep not in curve["epochs"]["registered"] or ep not in curve["epochs"]["D4"]:
        continue
    d = curve["epochs"]["D4"][ep]["losses"] - curve["epochs"]["registered"][ep]["losses"]
    for name, mk in [("overall", np.ones(len(d), bool))] + [(k, m) for k, m in bands.items() if m.any()] + list(groups.items()):
        m_, lo, hi = cluster_ci(d[mk], fid[mk])
        comp.append({"epoch": ep, "group": name, "targets": int(mk.sum()), "registered": float(curve["epochs"]["registered"][ep]["losses"][mk].mean()),
                     "D4": float(curve["epochs"]["D4"][ep]["losses"][mk].mean()), "d_mean": m_, "ci_lo": lo, "ci_hi": hi})
print("\npaired comparison on cleaned data, D4 minus registered; negative favors D4; 95% interval by filing:")
print(pd.DataFrame(comp).to_string(index=False))
pd.reset_option("display.float_format")

[edgar_xbrl_parser.py] Applied dynamic namespace mapping patch to xbrl.taxonomy and xbrl.instance
MC3ae contract f1f3d18f90b0b97f | D5 facts 1d6ee181e9fd1807 | contexts bb82e1d833c2983b
epoch: 126,767 microbatches, 1,234,902 targets; single 61,105, small 29,405, medium 4,794, full 31,463
registered rule: update per microbatch, warmup 12,677 updates, cosine to 1,014,136 | D4: step at >= 32 targets, warmup 123,490 targets, cosine to 9,879,216
cleaned held out targets 136,446: zero 0.2346 | concept_median 0.2242 | 0 to 1 119,861 | 1 to 3 14,846 | 3 to 10 1,739 | 10+ 0
[registered] init: heldout 0.2362
[registered] epoch 1 10%: steps 12,677 | heldout 0.2251 | 0 to 1 0.0739 | seen_common 0.2220 | d_pred_shuffle 0.0008213 | lr 3.00e-04 | 0.07 h
[registered] epoch 1 20%: steps 25,353 | heldout 0.2228 | 0 to 1 0.0735 | seen_common 0.2197 | d_pred_shuffle 1.376e-05 | lr 3.00e-04 | 0.13 h
[registered] epoch 1 30%: steps 38,030 | heldout 0.2226 | 0 to 1 0.0729 | seen_common 0.2192 | d_pred_shuffl

## MC3ae findings. D4 is the update rule on cleaned data

Contract `f1f3d18f90b0b97f`; D5 facts `1d6ee181e9fd1807`; contexts `bb82e1d833c2983b`. One seed, two epochs per arm, same initialization, same D3 interleaved sequences. Actual runtime 1.17 h (registered) and 0.73 h (D4), 1.90 h total.

| Arm, epoch 2 | heldout | 0 to 1 | 1 to 3 | 3 to 10 | seen_common | seen_rare | unseen | d_pred_shuffle |
|---|---|---|---|---|---|---|---|---|
| zero predictor | 0.2346 | | | | | | | |
| concept median | 0.2242 | | | | | | | |
| registered | 0.2188 | 0.0727 | 1.054 | 3.155 | 0.2150 | 0.2223 | 0.2668 | 0.0005 |
| D4 | 0.1665 | 0.0545 | 0.786 | 2.60 | 0.1583 | 0.1908 | 0.2307 | 0.129 |

**Paired, D4 minus registered, epoch 2 (95% interval by filing).** Overall −0.0523 [−0.0564, −0.0486]. Every band and every concept group excludes 0 in favor of D4: 0 to 1 −0.0182 [−0.0194, −0.0171]; 1 to 3 −0.269; 3 to 10 −0.555; seen_common −0.0567; seen_rare −0.0315; unseen −0.0361. At epoch 1 the 0 to 1 interval included 0 (−0.0002 [−0.0016, 0.0012]); the bulk band advantage appears in the second epoch.

**Reading.**

- The registered rule reproduces the context blind outcome of G1 on cleaned data: 0.005 below the concept median after two epochs, shuffle probe never above 0.0012. Its result is complete and is the register's record of the registered rule at full scale.
- D4 reads context (probe 0.13 to 0.25 through epoch 2) and is 0.058 below the concept median overall. Unlike the uncleaned run (MC3aa), the gain includes the 0 to 1 band, where 88% of the held out targets sit.
- Criterion C2 (reads context) is met. C1 (beats the concept median by paired interval) and C3 (no harm in 0 to 1 against the concept median) are closed in MC3af from the saved per target losses. C4: D4 is the rule for the W comparison.
- Limits: one seed; the probe uses 20 filings; the D4 epoch 2 curve moved from 0.1631 at 80% to 0.1665 at 100%, so the end of epoch figure is not the arm's minimum; the artifact rule's effect by issuer size is unverified (reporting by market cap and price at filing time is a queued item).

**Register.** D4 adopted as the update rule for all remaining pilot arms on D5 data. D5 and D4 together replace the uncleaned MC3z/MC3aa evidence, which is retained as history only.

## MC3af. C1 and C3 against the concept median, and the decile report (E1)

Reads `mc3ae_curve.pkl.lz4` and the D5 pilot file; no training. Builds the concept median predictor from training facts, then reports paired intervals of each arm against it, and adds the equal count report: ten bins on |signed log y| with edges at the training target deciles, held fixed for every later arm.

**Printed: `DECILE EDGES`:** one row per bin, `bin`, `lo_signed_log`, `hi_signed_log` (edges on |y| in signed log units; right edge inclusive, so ties at an edge fall in the lower bin), `lo_value`, `hi_value` (the same edges as stored pct change in fraction units, |v| = exp(edge) − 1), `train_share` (share of training targets in the bin; equal to 0.1 unless edges tie), `heldout_n`.

**Printed: `HELDOUT LOSS BY DECILE`:** per bin, `zero`, `concept_median`, `registered_ep2`, `D4_ep2`, and `D4_minus_median` with `ci_lo`, `ci_hi` (paired, filing clustered, 95%).

**Printed: `C1 AND C3 (paired against concept median)`:** per arm and epoch, rows for overall, the original bins (retained for continuity), and the concept groups: `arm_loss`, `median_loss`, `d_mean` (arm minus median; negative favors the arm), `ci_lo`, `ci_hi`, `pass` (interval entirely below 0).

Writes `E:\apriori-ml\mc3\mc3_decile_edges.pkl.lz4` (edges, value boundaries, training fingerprint) for reuse by every later arm.

In [3]:
# MC3af. C1 and C3 against the concept median; bins with fixed training edges (E1). Final per review.
import os, sys
os.environ["ENV"] = "prod"
for p_ in (r"E:\apriori-ml", r"C:\Users\jakem\Documents\Belk\Sec\Bloomberg Pipeline v1\BB_Pipeline_v15"):
    if p_ not in sys.path:
        sys.path.insert(0, p_)
import numpy as np
import pandas as pd
import torch
from set_attention_model import signed_log
from apriori.serialization import load_from_pickle, save_to_pickle

OUT_DIR = r"E:\apriori-ml\mc3"
PILOT, CURVE = os.path.join(OUT_DIR, "mc3_pilot_data_d5.pkl.lz4"), os.path.join(OUT_DIR, "mc3ae_curve.pkl.lz4")
EDGES_OUT = os.path.join(OUT_DIR, "mc3_decile_edges.pkl.lz4")
FACTS_FP, CONTRACT = "1d6ee181e9fd1807", "f1f3d18f90b0b97f"
RULES, EPOCHS, SEED, BOOT, N_BINS = ["registered", "D4"], [1, 2], 367, 2000, 10
BANDS = [(0, 1, "0 to 1"), (1, 3, "1 to 3"), (3, 10, "3 to 10")]

def huber(pred, y):
    r = np.abs(pred - y)
    return np.where(r <= 1.0, 0.5 * r * r, r - 0.5)

def cluster_ci(d, f):
    if len(d) == 0:
        return float("nan"), float("nan"), float("nan")
    uniq, inv = np.unique(f, return_inverse=True)
    s, c = np.bincount(inv, weights=d), np.bincount(inv)
    idx = np.random.default_rng((SEED, 91)).integers(0, len(uniq), size=(BOOT, len(uniq)))
    est = s[idx].sum(1) / c[idx].sum(1)
    return float(d.mean()), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

pdat = load_from_pickle(PILOT)
assert pdat["fingerprints"]["facts"] == FACTS_FP
curve = load_from_pickle(CURVE)
assert curve["contract"] == CONTRACT, f"curve contract {curve['contract']} is not {CONTRACT}"
indptr, indices, data, split = pdat["indptr"], pdat["indices"].astype(np.int64), pdat["data"], np.asarray(pdat["split"])
sl = signed_log(torch.as_tensor(np.asarray(data, dtype=np.float32))).numpy().astype(np.float64)
assert np.isfinite(sl).all(), "nonfinite signed log value in pilot data"
fact_row = np.repeat(np.arange(len(split)), np.diff(indptr))
tr_idx, ho_idx = np.flatnonzero(split == "train"), np.flatnonzero(split == "holdout")
tr_rows = np.concatenate([np.arange(int(indptr[i]), int(indptr[i + 1])) for i in tr_idx])
ho_rows = np.concatenate([np.arange(int(indptr[i]), int(indptr[i + 1])) for i in ho_idx])
tr_c, tr_y, tr_f = indices[tr_rows], sl[tr_rows], fact_row[tr_rows]
y_ho, c_ho, fid = curve["y_ho"], curve["c_ho"], curve["fid"]
assert len(y_ho) == len(ho_rows), "held out count differs from the curve file"
assert np.array_equal(c_ho, indices[ho_rows]), "held out concept identity differs from the curve file"
assert np.array_equal(fid, fact_row[ho_rows]), "held out filing identity differs from the curve file"
assert np.array_equal(y_ho, sl[ho_rows]), "held out target values differ from the curve file"

# ----- references on held out; concept frequency as distinct training filings -----
med = pd.Series(tr_y).groupby(tr_c).median()
global_median = float(np.median(tr_y))                     # fallback for concepts with no training occurrence
c_med = np.full(int(indices.max()) + 1, global_median); c_med[med.index.values] = med.values
l_zero, l_med = huber(np.zeros_like(y_ho), y_ho), huber(c_med[c_ho], y_ho)
occ_freq = np.bincount(tr_c, minlength=int(indices.max()) + 1)
pairs = np.unique(tr_f.astype(np.int64) * (int(indices.max()) + 1) + tr_c)
doc_freq = np.bincount(pairs % (int(indices.max()) + 1), minlength=int(indices.max()) + 1)
dup = int((occ_freq - doc_freq).sum())
print(f"concept frequency: distinct training filings per concept; {dup:,} repeated (filing, concept) occurrences"
      + (" (none; occurrence counts equal filing counts)" if dup == 0 else " (groups below use filing counts; MC3ae groups used occurrence counts)"))
df_ho = doc_freq[c_ho]
ay = np.abs(y_ho)
groups = {"overall": np.ones(len(y_ho), bool), **{nm: (ay >= lo) & (ay < hi) for lo, hi, nm in BANDS},
          "seen_common": df_ho >= 50, "seen_rare": (df_ho >= 1) & (df_ho < 50), "unseen": df_ho == 0}

# ----- saved losses: shape and finiteness -----
L = {(r, e): np.asarray(curve["epochs"][r][e]["losses"], dtype=np.float64) for r in RULES for e in EPOCHS if e in curve["epochs"][r]}
for key, loss in L.items():
    assert loss.shape == y_ho.shape, key
    assert np.isfinite(loss).all(), key
for name, loss in (("zero", l_zero), ("concept_median", l_med)):
    assert loss.shape == y_ho.shape and np.isfinite(loss).all(), name

# ----- bins on |y| from training quantiles; assignment edges separate from physical ranges -----
tr_ay = np.abs(tr_y)
q = np.quantile(tr_ay, np.linspace(0, 1, N_BINS + 1))
inner = np.unique(q[1:-1])                       # interior cut points; ties merge bins
if len(inner) < N_BINS - 1:
    print(f"note: {N_BINS - 1 - len(inner)} tied interior quantile edge(s) merged; {len(inner) + 1} bins (counts will not be equal)")
assign_edges = np.concatenate(([-np.inf], inner, [np.inf]))      # bin b: assign_edges[b] < |y| <= assign_edges[b+1]
nb = len(assign_edges) - 1
tr_bin = np.searchsorted(assign_edges, tr_ay, side="left") - 1
ho_bin = np.searchsorted(assign_edges, ay, side="left") - 1
assert tr_bin.min() >= 0 and tr_bin.max() < nb and ho_bin.min() >= 0 and ho_bin.max() < nb, "bin assignment out of range"
range_lo = np.concatenate(([0.0], inner))                          # physical magnitude ranges in signed log units
range_hi = np.concatenate((inner, [np.inf]))                       # top bin unbounded above
train_max_abs = float(tr_ay.max())
save_to_pickle({"assign_edges_signed_log": assign_edges, "range_lo_signed_log": range_lo, "range_hi_signed_log": range_hi,
                "range_lo_value": np.expm1(range_lo), "range_hi_value": np.expm1(range_hi), "train_max_abs_signed_log": train_max_abs,
                "n_bins": nb, "facts_fp": FACTS_FP,
                "rule": "bin b holds assign_edges[b] < |signed log y| <= assign_edges[b+1]; cut points are interior training quantiles with ties merged; top bin unbounded above"},
               EDGES_OUT, lz4_compression=True)

pd.set_option("display.float_format", lambda v: f"{v:.4g}")
rows = [{"bin": b, "assign_lo": assign_edges[b], "assign_hi": assign_edges[b + 1], "range_lo_signed_log": range_lo[b], "range_hi_signed_log": range_hi[b],
         "range_lo_value": np.expm1(range_lo[b]), "range_hi_value": np.expm1(range_hi[b]), "train_n": int((tr_bin == b).sum()),
         "train_share": float((tr_bin == b).mean()), "heldout_n": int((ho_bin == b).sum())} for b in range(nb)]
print(f"BIN EDGES (training |signed log y| quantiles; assignment open below, closed above; largest training |y| {train_max_abs:.4g})")
print(pd.DataFrame(rows).to_string(index=False))

rows = []
for b in range(nb):
    m = ho_bin == b
    dm, lo, hi = cluster_ci((L[("D4", 2)] - l_med)[m], fid[m])
    rows.append({"bin": b, "heldout_n": int(m.sum()), "zero": l_zero[m].mean() if m.any() else np.nan, "concept_median": l_med[m].mean() if m.any() else np.nan,
                 "registered_ep2": L[("registered", 2)][m].mean() if m.any() else np.nan, "D4_ep2": L[("D4", 2)][m].mean() if m.any() else np.nan,
                 "D4_minus_median": dm, "ci_lo": lo, "ci_hi": hi, "estimable": bool(m.any())})
print("\nHELDOUT LOSS BY BIN (D4 minus concept median, paired, 95% by filing; negative favors D4)")
print("bins are defined using realized target magnitude; within bin results do not define an inference time switching rule")
print(pd.DataFrame(rows).to_string(index=False))

rows = []
for (r, e), l in L.items():
    for name, m in groups.items():
        dm, lo, hi = cluster_ci((l - l_med)[m], fid[m])
        rows.append({"arm": r, "epoch": e, "group": name, "targets": int(m.sum()), "arm_loss": l[m].mean() if m.any() else np.nan,
                     "median_loss": l_med[m].mean() if m.any() else np.nan, "d_mean": dm, "ci_lo": lo, "ci_hi": hi,
                     "superior": bool(hi < 0) if np.isfinite(hi) else None})
print("\nC1 AND C3 (superiority against the concept median; superior = interval entirely below 0)")
print(pd.DataFrame(rows).to_string(index=False))
pd.reset_option("display.float_format")
print(f"\nsaved: {EDGES_OUT}")

concept frequency: distinct training filings per concept; 0 repeated (filing, concept) occurrences (none; occurrence counts equal filing counts)
BIN EDGES (training |signed log y| quantiles; assignment open below, closed above; largest training |y| 4.615)
 bin  assign_lo  assign_hi  range_lo_signed_log  range_hi_signed_log  range_lo_value  range_hi_value  train_n  train_share  heldout_n
   0       -inf    0.02933                    0              0.02933               0         0.02976   123491          0.1      15290
   1    0.02933    0.07506              0.02933              0.07506         0.02976         0.07795   123490          0.1      16274
   2    0.07506     0.1395              0.07506               0.1395         0.07795          0.1497   123491          0.1      15752
   3     0.1395      0.229               0.1395                0.229          0.1497          0.2573   123490          0.1      14774
   4      0.229     0.3521                0.229               0.3521      

## MC3ag0. Pre run certification for the W arm

Three checks, no training, about two minutes.

**Printed: `W0 CONTRACT RECOMPUTED`:** the MC3ae contract rebuilt from its saved parameters, `mc3ae-1`, and the current module sources, config, split and contexts. `match` must be True: it certifies that the modules, MCConfig and data MC3ae ran with are the ones on disk now, so the seeded construction and the seeded epoch plans are the same ones. MC3ae did not save an initial state hash, so this is the available proof of shared initialization; `init_state_sha16` is recorded now for every later arm.

**Printed: `CONSTRUCTION DETERMINISM`:** two seeded constructions compared tensor by tensor; `identical` must be True.

**Printed: `EPOCH PLANS AND ACCUMULATION GROUPS`:** per epoch, `plan_sha16`, `microbatches`, `groups_computed` (steps implied by the plan under the 32 target rule) and `W0_steps` (from the MC3ae curve); `match` must be True for both epochs.

**Printed: `BATCH_LOSS RETURNS (W, first microbatch)`:** index, type, shape or value of each item `batch_loss` returns, so the sampler's P and zero usable target counts can be recorded in MC3ag.

**Printed: `OBJECTIVE CHECK (first accumulation group)`:** `microbatches`, `targets`, loss under the level walk and the jump walk, `loss_abs_diff`, `grad_max_abs_diff`, `grad_max_rel_diff` (over all parameters, relative to the level walk gradient norm), `levels_s`, `jump_s`. This certifies that the jump path reproduces the certified level path through the whole W objective and its gradients, including the Horvitz Thompson term and the target averaging inside `batch_loss`. It does not re derive the estimator itself.

In [4]:
# MC3ag0. Pre run certification for W: W0 contract recompute, construction determinism, plan and group match, objective check.
import os, sys, time, math, hashlib, json, inspect
from dataclasses import asdict
os.environ["ENV"] = "prod"
for p_ in (r"E:\apriori-ml", r"C:\Users\jakem\Documents\Belk\Sec\Bloomberg Pipeline v1\BB_Pipeline_v15"):
    if p_ not in sys.path:
        sys.path.insert(0, p_)
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import importlib
import set_attention_cache, set_attention_model
import masked_concept_model as mcm
import mc_pilot_data as mpd
import mc_batched as mcb
importlib.reload(mcm); importlib.reload(mpd); importlib.reload(mcb)
from apriori.serialization import load_from_pickle

OUT_DIR = r"E:\apriori-ml\mc3"
PILOT, CTXF = os.path.join(OUT_DIR, "mc3_pilot_data_d5.pkl.lz4"), os.path.join(OUT_DIR, "mc3_contexts_d5.pkl.lz4")
W0_CURVE = os.path.join(OUT_DIR, "mc3ae_curve.pkl.lz4")
FACTS_FP, W0_CONTRACT, W0_HARNESS = "1d6ee181e9fd1807", "f1f3d18f90b0b97f", "mc3ae-1"
KIND, M_W, RUN_FACTOR, N_T, BATCH, MASK_RATE, ORDER_SEED_TAG, MIN_TARGETS, SEED = "W", 2048, 1, 12, 32, 0.15, 131, 32, 367
dev, f32 = "cuda", torch.float32

def sha16(b):
    return hashlib.sha256(b).hexdigest()[:16]

pdat = load_from_pickle(PILOT)
assert pdat["fingerprints"]["facts"] == FACTS_FP
cdat = load_from_pickle(CTXF)
assert cdat["pilot_facts_fp"] == FACTS_FP
w0 = load_from_pickle(W0_CURVE)
assert w0["contract"] == W0_CONTRACT
indptr, indices, data, split = pdat["indptr"], pdat["indices"], pdat["data"], np.asarray(pdat["split"])
NT = torch.from_numpy(pdat["name_table"].astype(np.float32)).to(dev)
len_a = np.diff(indptr)
fact_row = np.repeat(np.arange(len(split)), len_a)
fact_local = np.arange(len(indices)) - indptr[fact_row]
tr_fact = np.isin(fact_row, np.flatnonzero(split == "train"))
occ_filing, occ_local, occ_concept = fact_row[tr_fact], fact_local[tr_fact], indices[tr_fact]

# ----- 1. W0 contract recomputed from its saved parameters and the current sources -----
sources = {m.__name__: sha16(inspect.getsource(m).encode()) for m in (set_attention_cache, set_attention_model, mcm, mpd, mcb)}
recomputed = sha16(json.dumps({"sources": sources, "facts_fp": FACTS_FP, "contexts_fp": cdat["fingerprint"], "split": sha16(split.astype("U").tobytes()),
                               "mc_config": asdict(mcm.MCConfig()), "params": w0["params"], "harness_version": W0_HARNESS},
                              sort_keys=True, default=str).encode())
torch.manual_seed(SEED); m1 = mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32)
torch.manual_seed(SEED); m2 = mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32)
same = all(torch.equal(a, b) for (ka, a), (kb, b) in zip(m1.state_dict().items(), m2.state_dict().items()) if ka == kb)
init_sha = sha16(b"".join(v.detach().cpu().numpy().tobytes() for v in m1.state_dict().values()))
print("W0 CONTRACT RECOMPUTED")
print(pd.DataFrame([{"recomputed": recomputed, "W0_contract": W0_CONTRACT, "match": recomputed == W0_CONTRACT, "init_state_sha16": init_sha}]).to_string(index=False))
print("\nCONSTRUCTION DETERMINISM")
print(pd.DataFrame([{"identical": same, "n_params": sum(p.numel() for p in m1.parameters())}]).to_string(index=False))
del m2

# ----- 2. epoch plans and accumulation groups against W0 -----
def epoch_keys(e):
    plan = mpd.column_epoch_plan(occ_concept, e, SEED, BATCH)
    order = np.random.default_rng((SEED, e, ORDER_SEED_TAG)).permutation(len(plan))
    plan = [plan[i] for i in order]
    return [[(int(occ_filing[o]), e, 1, (int(occ_local[o]),)) for o in occ] for _, occ in plan]

w0_steps = {1: w0["epochs"]["D4"][1]["row"]["opt_steps"], 2: w0["epochs"]["D4"][2]["row"]["opt_steps"] - w0["epochs"]["D4"][1]["row"]["opt_steps"]}
rows, keys_by_epoch = [], {}
for e in (0, 1):
    keys = epoch_keys(e)
    keys_by_epoch[e] = keys
    sizes = np.array([len(k) for k in keys])
    acc, groups_ = 0, 0
    for s in sizes:
        acc += int(s)
        if acc >= MIN_TARGETS:
            groups_ += 1; acc = 0
    if acc > 0:
        groups_ += 1
    rows.append({"epoch": e + 1, "plan_sha16": sha16(np.concatenate([np.array([k[0] for k in mb] + [k[3][0] for k in mb]) for mb in keys]).astype(np.int64).tobytes()),
                 "microbatches": len(keys), "targets": int(sizes.sum()), "groups_computed": groups_, "W0_steps": w0_steps[e + 1], "match": groups_ == w0_steps[e + 1]})
print("\nEPOCH PLANS AND ACCUMULATION GROUPS")
print(pd.DataFrame(rows).to_string(index=False))

# ----- 3. objective check on the first accumulation group of epoch 1 -----
ds = mpd.PilotDataset(indptr, indices, data, pdat["file_names"], cdat["ctx"][N_T], KIND, M=M_W, run_factor=RUN_FACTOR, seed=SEED, rate=MASK_RATE)
group, acc = [], 0
for mb in keys_by_epoch[0]:
    group.append(mb); acc += len(mb)
    if acc >= MIN_TARGETS:
        break
loader = DataLoader(ds, batch_sampler=group, collate_fn=mpd.collate_list, num_workers=0)
batches = list(loader)
out0 = mcb.batch_loss(m1, KIND, batches[0], NT, dev, walk="levels")
print("\nBATCH_LOSS RETURNS (W, first microbatch)")
desc = []
for j, o in enumerate(out0 if isinstance(out0, (tuple, list)) else (out0,)):
    if torch.is_tensor(o):
        desc.append({"index": j, "type": "tensor", "shape": tuple(o.shape), "value": float(o) if o.numel() == 1 else ""})
    elif isinstance(o, dict):
        desc.append({"index": j, "type": "dict", "shape": "", "value": {k: (tuple(v.shape) if torch.is_tensor(v) else v) for k, v in o.items()}})
    else:
        desc.append({"index": j, "type": type(o).__name__, "shape": getattr(o, "shape", ""), "value": o if np.isscalar(o) else ""})
print(pd.DataFrame(desc).to_string(index=False))
print(f"first microbatch item keys: {sorted(batches[0][0].keys())}")
del out0

res = {}
for walk in ("levels", "jump"):
    torch.manual_seed(SEED); m = mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32)
    m.train(); m.zero_grad(set_to_none=True)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    tot = 0.0
    for items in batches:
        n = len(items)
        loss = mcb.batch_loss(m, KIND, items, NT, dev, walk=walk)[0]
        (loss * n).backward()
        tot += float(loss) * n
    torch.cuda.synchronize()
    res[walk] = {"loss_sum": tot, "grads": [p.grad.detach().double().clone() if p.grad is not None else None for p in m.parameters()],
                 "seconds": time.perf_counter() - t0}
    del m
gl, gj = res["levels"]["grads"], res["jump"]["grads"]
assert all((a is None) == (b is None) for a, b in zip(gl, gj)), "gradient presence differs between walks"
diffs = [(a - b).abs().max().item() for a, b in zip(gl, gj) if a is not None]
norm = math.sqrt(sum((a ** 2).sum().item() for a in gl if a is not None))
print("\nOBJECTIVE CHECK (first accumulation group)")
print(pd.DataFrame([{"microbatches": len(batches), "targets": sum(len(b) for b in batches), "loss_sum_levels": res["levels"]["loss_sum"], "loss_sum_jump": res["jump"]["loss_sum"],
                     "loss_abs_diff": abs(res["levels"]["loss_sum"] - res["jump"]["loss_sum"]), "grad_max_abs_diff": max(diffs), "grad_max_rel_diff": max(diffs) / max(norm, 1e-30),
                     "grad_norm_levels": norm, "levels_s": res["levels"]["seconds"], "jump_s": res["jump"]["seconds"]}]).to_string(index=False))

W0 CONTRACT RECOMPUTED
      recomputed      W0_contract  match init_state_sha16
f1f3d18f90b0b97f f1f3d18f90b0b97f   True 88071b7a2e52af9f

CONSTRUCTION DETERMINISM
 identical  n_params
      True   1647627

EPOCH PLANS AND ACCUMULATION GROUPS
 epoch       plan_sha16  microbatches  targets  groups_computed  W0_steps  match
     1 c9ec2fc1ce2e0860        126767  1234902            32995     32995   True
     2 8aef1402fd7de8d8        126767  1234902            32958     32958   True

BATCH_LOSS RETURNS (W, first microbatch)
 index   type  shape     value
     0 tensor     ()  0.263136
     1    int              988
     2 tensor   (1,)  0.034076
     3 tensor (987,)          
first microbatch item keys: ['P', 'cols', 'i', 'n', 'plan', 'tgt', 'trie', 'vals']

OBJECTIVE CHECK (first accumulation group)
 microbatches  targets  loss_sum_levels  loss_sum_jump  loss_abs_diff  grad_max_abs_diff  grad_max_rel_diff  grad_norm_levels  levels_s   jump_s
            6       37         7.237445     

## MC3ag. W under D4 on cleaned data against the frozen W0 (primary endpoint: epoch 2 versus epoch 2)

**What runs.** Arm W (subset supervision through the cache, Horvitz Thompson subset term, requested subset budget M 2,048 per filing draw, pointer jumping walk per D1; MC3ag0 certified the jump path against the level path through the full objective) trained on the D5 cleaned data under D4: same initialization as MC3ae (state hash `88071b7a2e52af9f`, asserted), same D3 interleaved epoch sequences (plan hashes asserted), same accumulation threshold (32 targets, step counts asserted against W0), same target exposure schedule, two epochs. Scored on the same cleaned held out targets with full context, leave one out. Compared target by target with the saved W0 losses from `mc3ae_curve.pkl.lz4`.

**Resume rule.** The checkpoint is authoritative: it carries the curve state, and the curve file is a copy written atomically after it.

**Sampler accounting.** `P` is the sampler's actual subset count per filing draw (requested M 2,048). `usable_frac` is exact on every microbatch: usable subsets (from `batch_loss`) over the sum of `P`. Per target usable counts and the share of targets with none are exact on every 64th microbatch (the forward pass and walk are repeated under no grad on those), so `zero_frac`, `u_med` and `u_min` are sampled statistics.

**Printed: header:** contract, W0 contract, the assertions passed, and the initial held out loss.

**Printed at each 10% boundary and epoch end:** `epoch`, `pct`, `opt_steps`, `heldout`, `0 to 1`, `seen_common`, `d_pred_shuffle`, `P_med`, `P_min`, `usable_frac`, `zero_frac`, `u_med`, `u_min`, `lr`, `hours`. Descriptive; no interim checkpoint is compared to W0's endpoint.

**Printed at the end**

- `MC3ag W UNDER D4 BY EPOCH`: `heldout`, bands, groups, `d_pred_shuffle`, `opt_steps`, `hours`, `P_med`, `P_min`, `usable_frac`, `zero_frac` (sampled), `u_med` (sampled), `sampled_targets`, `grad_max_windows`, `upd_med_windows`, `peak_gib`, with the W0 rows for reference.
- `PRIMARY ENDPOINT`: W epoch 2 minus W0 epoch 2 on the pooled mean, paired, filing clustered 95% interval; `verdict` is "W better" if the interval is entirely below 0, "W0 better" if entirely above, otherwise "no difference established".
- `TRADEOFFS (reported, not gating)`: the same difference by the ten fixed bins, the retained magnitude bins, and the concept groups; epoch 1 rows descriptive.

Files under `E:\apriori-ml\mc3`: `mc3ag_W_last.pt`, `mc3ag_W_epoch{1,2}.pt`, `mc3ag_curve.pkl.lz4`. Expected about 7 h plus a few minutes per epoch for the sampled accounting.

In [7]:
# MC3ag. W under D4 on D5 data; two epochs; primary endpoint W epoch 2 vs W0 epoch 2 (MC3ae D4). Checkpoint authoritative; sampler accounting.
import os, sys, time, math, hashlib, json, inspect
from dataclasses import asdict
os.environ["ENV"] = "prod"
for p_ in (r"E:\apriori-ml", r"C:\Users\jakem\Documents\Belk\Sec\Bloomberg Pipeline v1\BB_Pipeline_v15"):
    if p_ not in sys.path:
        sys.path.insert(0, p_)
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import importlib
import set_attention_cache, set_attention_model
import masked_concept_model as mcm
import mc_pilot_data as mpd
import mc_batched as mcb
importlib.reload(mcm); importlib.reload(mpd); importlib.reload(mcb)
from set_attention_model import signed_log
from apriori.serialization import load_from_pickle, save_to_pickle

HARNESS_VERSION = "mc3ag-3"
OUT_DIR = r"E:\apriori-ml\mc3"
PILOT, CTXF = os.path.join(OUT_DIR, "mc3_pilot_data_d5.pkl.lz4"), os.path.join(OUT_DIR, "mc3_contexts_d5.pkl.lz4")
W0_CURVE, EDGES = os.path.join(OUT_DIR, "mc3ae_curve.pkl.lz4"), os.path.join(OUT_DIR, "mc3_decile_edges.pkl.lz4")
CURVE, LAST = os.path.join(OUT_DIR, "mc3ag_curve.pkl.lz4"), os.path.join(OUT_DIR, "mc3ag_W_last.pt")
CURVE_TMP = os.path.join(OUT_DIR, "mc3ag_curve_tmp.pkl.lz4")
FACTS_FP, W0_CONTRACT, W0_HARNESS = "1d6ee181e9fd1807", "f1f3d18f90b0b97f", "mc3ae-1"
INIT_SHA, PLAN_SHA = "88071b7a2e52af9f", {0: "c9ec2fc1ce2e0860", 1: "8aef1402fd7de8d8"}
KIND, WALK, EPOCHS_RUN, EPOCHS_REGISTERED = "W", "jump", 2, 8
M_W, RUN_FACTOR, N_T, BATCH, MASK_RATE, ORDER_SEED_TAG = 2048, 1, 12, 32, 0.15, 131
LR, WD, WARMUP_FRAC, MIN_TARGETS, SAMPLE_EVERY = 3e-4, 0.01, 0.10, 32, 64
WORKERS, SEED, EVAL_CHUNK, N_PROBE, LOADER_TIMEOUT, MAX_STALLS, BOOT = 16, 367, 1024, 20, 300, 20, 2000
BANDS = [(0, 1, "0 to 1"), (1, 3, "1 to 3"), (3, 10, "3 to 10")]
dev, f32 = "cuda", torch.float32

def sha16(b):
    return hashlib.sha256(b).hexdigest()[:16]

def huber(pred, y):
    r = np.abs(pred - y)
    return np.where(r <= 1.0, 0.5 * r * r, r - 0.5)

# ----- data (D5), the frozen W0, and the recomputed W0 contract -----
pdat = load_from_pickle(PILOT)
assert pdat["fingerprints"]["facts"] == FACTS_FP and "D5" in pdat.get("deviation", {})
cdat = load_from_pickle(CTXF)
assert cdat["pilot_facts_fp"] == FACTS_FP
w0 = load_from_pickle(W0_CURVE)
assert w0["contract"] == W0_CONTRACT, f"W0 curve contract {w0['contract']} is not {W0_CONTRACT}"
assert all(e in w0["epochs"]["D4"] for e in (1, 2)), "W0 D4 epochs 1 and 2 are required"
edges = load_from_pickle(EDGES)
assert edges["facts_fp"] == FACTS_FP
indptr, indices, data = pdat["indptr"], pdat["indices"], pdat["data"]
split = np.asarray(pdat["split"])
tr_idx, ho_idx = np.flatnonzero(split == "train"), np.flatnonzero(split == "holdout")
len_a = np.diff(indptr)
NT = torch.from_numpy(pdat["name_table"].astype(np.float32)).to(dev)
ctx12 = cdat["ctx"][N_T]
doc_freq = cdat["doc_freq_train"]
sl = signed_log(torch.as_tensor(np.asarray(data, dtype=np.float32))).numpy().astype(np.float64)
fact_row = np.repeat(np.arange(len(split)), len_a)
fact_local = np.arange(len(indices)) - indptr[fact_row]
tr_fact = np.isin(fact_row, tr_idx)
occ_filing, occ_local, occ_concept = fact_row[tr_fact], fact_local[tr_fact], indices[tr_fact]
N_OCC = len(occ_filing)
sources = {m.__name__: sha16(inspect.getsource(m).encode()) for m in (set_attention_cache, set_attention_model, mcm, mpd, mcb)}
split_sha = sha16(split.astype("U").tobytes())
w0_recomputed = sha16(json.dumps({"sources": sources, "facts_fp": FACTS_FP, "contexts_fp": cdat["fingerprint"], "split": split_sha,
                                  "mc_config": asdict(mcm.MCConfig()), "params": w0["params"], "harness_version": W0_HARNESS},
                                 sort_keys=True, default=str).encode())
assert w0_recomputed == W0_CONTRACT, f"W0 contract recomputed as {w0_recomputed}; modules, config or data differ from MC3ae"

def epoch_keys(e):
    plan = mpd.column_epoch_plan(occ_concept, e, SEED, BATCH)
    order = np.random.default_rng((SEED, e, ORDER_SEED_TAG)).permutation(len(plan))
    plan = [plan[i] for i in order]
    allocc = np.concatenate([occ for _, occ in plan])
    assert len(allocc) == N_OCC and np.array_equal(np.sort(allocc), np.arange(N_OCC)), "plan coverage"
    keys = [[(int(occ_filing[o]), e, 1, (int(occ_local[o]),)) for o in occ] for _, occ in plan]
    h = sha16(np.concatenate([np.array([k[0] for k in mb] + [k[3][0] for k in mb]) for mb in keys]).astype(np.int64).tobytes())
    assert h == PLAN_SHA[e], f"epoch {e + 1} plan hash {h} differs from the certified {PLAN_SHA[e]}"
    return keys

UPDATES_PER_EPOCH = len(epoch_keys(0))
BOUNDS = [int(round(UPDATES_PER_EPOCH * k / 10)) for k in range(1, 11)]
WARM_T, TOTAL_T = WARMUP_FRAC * N_OCC, N_OCC * EPOCHS_REGISTERED
assert w0["params"]["D4"]["warmup_targets"] == WARM_T and w0["params"]["D4"]["total_targets"] == TOTAL_T, "schedule differs from W0"
W0_STEPS = {1: w0["epochs"]["D4"][1]["row"]["opt_steps"], 2: w0["epochs"]["D4"][2]["row"]["opt_steps"]}

def lr_d4(t):
    return LR * t / WARM_T if t < WARM_T else LR * 0.5 * (1 + math.cos(math.pi * min(t - WARM_T, TOTAL_T - WARM_T) / max(TOTAL_T - WARM_T, 1)))

# ----- held out targets, identical to W0 -----
ho_rows = np.concatenate([np.arange(int(indptr[i]), int(indptr[i + 1])) for i in ho_idx])
y_ho, c_ho, fid = sl[ho_rows], indices[ho_rows].astype(np.int64), fact_row[ho_rows]
assert np.array_equal(y_ho, w0["y_ho"]) and np.array_equal(c_ho, w0["c_ho"]) and np.array_equal(fid, w0["fid"]), "held out set differs from W0"
ay = np.abs(y_ho)
bands = {nm: (ay >= lo) & (ay < hi) for lo, hi, nm in BANDS}
df_ho = doc_freq[c_ho]
groups = {"seen_common": df_ho >= 50, "seen_rare": (df_ho >= 1) & (df_ho < 50), "unseen": df_ho == 0}
ho_bin = np.searchsorted(edges["assign_edges_signed_log"], ay, side="left") - 1
bins = {f"bin {b}": ho_bin == b for b in range(int(edges["n_bins"]))}
probe_f = np.sort(np.random.default_rng((SEED, 181)).choice(ho_idx[len_a[ho_idx] <= 1024], N_PROBE, replace=False))
probe = []
for i in probe_f:
    lo, hi = int(indptr[i]), int(indptr[i + 1])
    raw = torch.as_tensor(np.asarray(data[lo:hi], dtype=np.float32), device=dev)
    perm = torch.as_tensor(np.random.default_rng((SEED, 171, int(i))).permutation(hi - lo), device=dev)
    probe.append((NT[torch.as_tensor(indices[lo:hi].astype(np.int64), device=dev)], raw, raw[perm]))

PARAMS = {"arm": KIND, "walk": WALK, "data": "D5", "order": "interleaved", "order_seed_tag": ORDER_SEED_TAG, "M_W_requested": M_W, "run_factor": RUN_FACTOR,
          "batch": BATCH, "epochs_registered": EPOCHS_REGISTERED, "epochs_run": EPOCHS_RUN, "lr": LR, "wd": WD,
          "D4": {"min_targets_per_step": MIN_TARGETS, "warmup_targets": WARM_T, "total_targets": TOTAL_T, "lr_by": "target exposure, group end"},
          "seed": SEED, "init_state_sha16": INIT_SHA, "plan_sha16": PLAN_SHA, "sampler_accounting_every": SAMPLE_EVERY,
          "eval": "cleaned heldout, all facts, leave one out, full context",
          "compared_to": W0_CONTRACT, "primary_endpoint": "W epoch 2 vs W0 epoch 2, pooled mean Huber, paired by filing"}
CONTRACT = sha16(json.dumps({"sources": sources, "facts_fp": FACTS_FP, "contexts_fp": cdat["fingerprint"], "split": split_sha,
                             "mc_config": asdict(mcm.MCConfig()), "params": PARAMS, "harness_version": HARNESS_VERSION},
                            sort_keys=True, default=str).encode())
ds = mpd.PilotDataset(indptr, indices, data, pdat["file_names"], ctx12, KIND, M=M_W, run_factor=RUN_FACTOR, seed=SEED, rate=MASK_RATE)
plan_fn, walk_fn = mcb.WALKS[WALK]

def new_model():
    torch.manual_seed(SEED)
    m = mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32)
    h = sha16(b"".join(v.detach().cpu().numpy().tobytes() for v in m.state_dict().values()))
    assert h == INIT_SHA, f"initial state hash {h} differs from the certified {INIT_SHA}"
    return m

@torch.no_grad()
def usable_per_target(model, items):
    """Exact per target usable subset counts for one microbatch: repeats the W forward pass and walk of batch_loss under no grad."""
    rc, rg = torch.get_rng_state(), torch.cuda.get_rng_state_all()
    model.eval()
    ns = np.array([it["n"] for it in items], dtype=np.int64)
    offs = np.concatenate([[0], np.cumsum(ns)[:-1]])
    cols = torch.as_tensor(np.concatenate([it["cols"] for it in items]), device=dev)
    vals = torch.as_tensor(np.concatenate([it["vals"] for it in items]), device=dev).to(NT.dtype)
    nv = NT[cols]
    ch_item = np.concatenate([np.full(len(it["tgt"]), b) for b, it in enumerate(items)])
    ch_local = np.concatenate([np.asarray(it["tgt"], dtype=np.int64) for it in items])
    ch_global = offs[ch_item] + ch_local
    C = len(ch_global)
    z, V, _ = model.scores(nv, vals, nv[torch.as_tensor(ch_global, device=dev)])
    tries = [mcb._shift(items[int(ch_item[c])]["trie"], offs[int(ch_item[c])]) for c in range(C)]
    plan = plan_fn(tries)
    _, _, _, use = walk_fn(z, V, plan, owned=True, owner_target_pos=ch_global)
    owner = np.asarray(plan["term_owner"])
    u = use[:, 0].detach().cpu().numpy().astype(bool)
    model.train()
    torch.set_rng_state(rc); torch.cuda.set_rng_state_all(rg)
    return np.bincount(owner[u], minlength=C)

@torch.no_grad()
def predict_filing(model, nv, vals):
    n = nv.shape[0]
    preds = []
    for s in range(0, n, EVAL_CHUNK):
        tg = torch.arange(s, min(n, s + EVAL_CHUNK), device=dev)
        c = torch.arange(len(tg), device=dev)
        z, V, e = model.scores(nv, vals, nv[tg])
        member = torch.ones(len(tg), n, dtype=torch.bool, device=dev)
        member[c, tg] = False
        fp, fL, fk, fu = mcm.mc_masked_attention(z, V, member, c)
        assert bool(fu.all())
        preds.append(model.predict(fp, fL, fk, e))
    return torch.cat(preds).double().cpu().numpy()

@torch.no_grad()
def evaluate(model):
    t0 = time.perf_counter()
    rc, rg = torch.get_rng_state(), torch.cuda.get_rng_state_all()
    model.eval()
    out = []
    for i in ho_idx:
        lo, hi = int(indptr[i]), int(indptr[i + 1])
        out.append(predict_filing(model, NT[torch.as_tensor(indices[lo:hi].astype(np.int64), device=dev)],
                                  torch.as_tensor(np.asarray(data[lo:hi], dtype=np.float32), device=dev)))
    pred = np.concatenate(out)
    assert np.isfinite(pred).all()
    dp, pb = [], []
    for nv, raw, shuf in probe:
        p0, p1 = predict_filing(model, nv, raw), predict_filing(model, nv, shuf)
        dp.append(np.abs(p1 - p0)); pb.append(p0)
    model.train()
    torch.set_rng_state(rc); torch.cuda.set_rng_state_all(rg)
    l = huber(pred, y_ho)
    ev = {"heldout": float(l.mean()), **{k: float(l[m].mean()) if m.any() else float("nan") for k, m in bands.items()},
          **{k: float(l[m].mean()) for k, m in groups.items()},
          "d_pred_shuffle": float(np.median(np.concatenate(dp))), "pred_spread": float(np.std(np.concatenate(pb)))}
    return l, ev, time.perf_counter() - t0

def save_state(path, model, opt, st, curve):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "st": st, "curve": curve, "contract": CONTRACT,
                "rng_cpu": torch.get_rng_state(), "rng_cuda": torch.cuda.get_rng_state_all()}, path + ".tmp")
    os.replace(path + ".tmp", path)
    save_to_pickle(curve, CURVE_TMP, lz4_compression=True)
    os.replace(CURVE_TMP, CURVE)

def cluster_ci(d, f):
    if len(d) == 0:
        return float("nan"), float("nan"), float("nan")
    uniq, inv = np.unique(f, return_inverse=True)
    s, c = np.bincount(inv, weights=d), np.bincount(inv)
    idx = np.random.default_rng((SEED, 91)).integers(0, len(uniq), size=(BOOT, len(uniq)))
    est = s[idx].sum(1) / c[idx].sum(1)
    return float(d.mean()), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

def window_stats(st):
    P = np.array(st["P"]) if st["P"] else np.array([np.nan])
    su = np.array(st["samp_u"]) if st["samp_u"] else np.array([np.nan])
    return {"P_med": float(np.median(P)), "P_min": float(P.min()), "usable_frac": st["U"] / max(st["Psum"], 1),
            "zero_frac": float((su == 0).mean()) if st["samp_u"] else np.nan, "u_med": float(np.median(su)), "u_min": float(su.min()),
            "sampled_targets": len(st["samp_u"])}

print(f"MC3ag contract {CONTRACT} | compares to W0 contract {W0_CONTRACT} (recomputed, match) | D5 facts {FACTS_FP} | contexts {cdat['fingerprint']}")
print(f"initialization {INIT_SHA} asserted | epoch plans {PLAN_SHA[0]}, {PLAN_SHA[1]} asserted | W0 steps {W0_STEPS[1]:,} and {W0_STEPS[2] - W0_STEPS[1]:,}")
print(f"epoch: {UPDATES_PER_EPOCH:,} microbatches, {N_OCC:,} targets | D4: step at >= {MIN_TARGETS} targets, warmup {WARM_T:,.0f} targets, cosine to {TOTAL_T:,} | "
      f"W: requested M {M_W}, run_factor {RUN_FACTOR}, walk {WALK} | per target usable counts sampled every {SAMPLE_EVERY} microbatches")

model = new_model()
params = list(model.parameters())
opt = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location="cpu", weights_only=False)
    assert ck["contract"] == CONTRACT, f"{LAST} was written under contract {ck['contract']}; delete mc3ag_*.pt and {CURVE} to start over"
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); st, curve = ck["st"], ck["curve"]
    torch.set_rng_state(ck["rng_cpu"]); torch.cuda.set_rng_state_all(ck["rng_cuda"])
    print(f"resumed from checkpoint: epoch index {st['e']}, microbatch {st['k']:,}, steps {st['steps']:,} (curve taken from the checkpoint)")
else:
    assert not os.path.exists(CURVE), f"{CURVE} exists without a checkpoint; delete it to start over"
    _, ev0, t_ev = evaluate(model)
    st = {"e": 0, "k": 0, "t": 0, "steps": 0, "train_s": 0.0, "eval_s": t_ev, "gwin": [], "gmax": 0.0, "uwin": [], "ups": [],
          "tps": [], "peak": 0.0, "stalls": [], "pending_ckpt": False, "P": [], "U": 0, "Psum": 0, "samp_u": [], "Swin": []}
    curve = {"contract": CONTRACT, "params": PARAMS, "y_ho": y_ho, "c_ho": c_ho, "fid": fid, "rows": [], "epochs": {}}
    curve["rows"].append({"epoch": 0, "pct": 0, "opt_steps": 0, "lr": 0.0, **ev0, "hours": t_ev / 3600})
    print(f"init: heldout {ev0['heldout']:.4f}")
model.train()
torch.cuda.reset_peak_memory_stats()
acc_n = 0
opt.zero_grad(set_to_none=True)

def do_step(count, lr):
    for p in params:
        if p.grad is not None:
            p.grad.div_(count)
    grads = [p.grad for p in params if p.grad is not None]
    if not all(bool(torch.isfinite(q).all()) for q in grads):
        raise RuntimeError(f"nonfinite gradient at step {st['steps']}; stopped before the update")
    st["gmax"] = max(st["gmax"], float(torch.sqrt(sum((q.double() ** 2).sum() for q in grads))))
    for gr in opt.param_groups:
        gr["lr"] = lr
    before = [p.detach().clone() for p in params]
    opt.step()
    st["ups"].append(float(torch.sqrt(sum(((p.detach() - b).double() ** 2).sum() for p, b in zip(params, before)))))
    opt.zero_grad(set_to_none=True)
    st["steps"] += 1; st["tps"].append(count)

for e in range(st["e"], EPOCHS_RUN):
    if (e + 1) in curve["epochs"]:
        continue
    keys = epoch_keys(e)
    epoch_done = False
    while not epoch_done:
        loader = DataLoader(ds, batch_sampler=keys[st["k"]:], collate_fn=mpd.collate_list, num_workers=WORKERS, prefetch_factor=2, timeout=LOADER_TIMEOUT)
        t_seg = time.perf_counter()
        try:
            for items in loader:
                k = st["k"]
                assert [it["i"] for it in items] == [key[0] for key in keys[k]], "loader out of plan order"
                n = len(items)
                ps = [int(it["P"]) for it in items]
                st["P"].extend(ps); st["Psum"] += sum(ps)
                if k % SAMPLE_EVERY == 0:
                    st["samp_u"].extend(usable_per_target(model, items).tolist())
                loss, exposures, _, _ = mcb.batch_loss(model, KIND, items, NT, dev, walk=WALK)
                st["U"] += int(exposures) - n
                if not bool(torch.isfinite(loss)):
                    raise RuntimeError(f"nonfinite loss at epoch {e + 1} microbatch {k}")
                st["k"] += 1; st["t"] += n
                end = st["k"] == UPDATES_PER_EPOCH
                (loss * n).backward()
                acc_n += n
                if acc_n >= MIN_TARGETS or end:
                    do_step(acc_n, lr_d4(st["t"])); acc_n = 0
                    if st["pending_ckpt"]:
                        st["train_s"] += time.perf_counter() - t_seg; t_seg = time.perf_counter()
                        save_state(LAST, model, opt, st, curve); st["pending_ckpt"] = False
                if st["k"] in BOUNDS:
                    st["train_s"] += time.perf_counter() - t_seg
                    ups = np.array(st["ups"]) if st["ups"] else np.array([np.nan])
                    st["uwin"].append(round(float(np.median(ups)), 4)); st["ups"] = []
                    st["gwin"].append(round(st["gmax"], 2)); st["gmax"] = 0.0
                    ws = window_stats(st)
                    st["Swin"].append(ws); st["P"] = []; st["U"] = 0; st["Psum"] = 0; st["samp_u"] = []
                    st["peak"] = max(st["peak"], torch.cuda.max_memory_allocated() / 2**30)
                    pct = BOUNDS.index(st["k"]) * 10 + 10
                    l_new, ev, t_ev = evaluate(model)
                    st["eval_s"] += t_ev
                    lr_now = lr_d4(st["t"])
                    row = {"epoch": e + 1, "pct": pct, "opt_steps": st["steps"], "lr": lr_now, **ev, **ws, "hours": (st["train_s"] + st["eval_s"]) / 3600}
                    curve["rows"].append(row)
                    print(f"epoch {e + 1} {pct}%: steps {st['steps']:,} | heldout {ev['heldout']:.4f} | 0 to 1 {ev['0 to 1']:.4f} | seen_common {ev['seen_common']:.4f} | "
                          f"d_pred_shuffle {ev['d_pred_shuffle']:.4g} | P_med {ws['P_med']:.0f} | P_min {ws['P_min']:.0f} | usable_frac {ws['usable_frac']:.3f} | "
                          f"zero_frac {ws['zero_frac']:.4f} | u_med {ws['u_med']:.0f} | u_min {ws['u_min']:.0f} | lr {lr_now:.2e} | {row['hours']:.2f} h")
                    if end:
                        assert st["steps"] == W0_STEPS[e + 1], f"W steps {st['steps']:,} differ from W0 steps {W0_STEPS[e + 1]:,} at epoch {e + 1}"
                        tps = np.array(st["tps"])
                        curve["epochs"][e + 1] = {"losses": l_new, "row": row, "grad_max_windows": st["gwin"], "upd_med_windows": st["uwin"], "sampler_windows": st["Swin"],
                                                  "peak_gib": st["peak"], "targets_per_step": (float(np.median(tps)), float(np.percentile(tps, 90)), int(tps.max()))}
                        st.update({"e": e + 1, "k": 0, "gwin": [], "uwin": [], "tps": [], "Swin": [], "pending_ckpt": False})
                        save_state(os.path.join(OUT_DIR, f"mc3ag_W_epoch{e + 1}.pt"), model, opt, st, curve)
                        save_state(LAST, model, opt, st, curve)
                        epoch_done = True
                    elif acc_n == 0:
                        save_state(LAST, model, opt, st, curve)
                    else:
                        st["pending_ckpt"] = True
                    t_seg = time.perf_counter()
                    if epoch_done:
                        break
        except RuntimeError as ex:
            if "DataLoader timed out" not in str(ex):
                raise
            st["stalls"].append((e, st["k"]))
            print(f"loader stalled at epoch {e + 1} microbatch {st['k']:,}; rebuilding (stall {len(st['stalls'])} of at most {MAX_STALLS})")
            if len(st["stalls"]) >= MAX_STALLS:
                raise RuntimeError(f"{MAX_STALLS} loader stalls; stopping") from None
            st["train_s"] += time.perf_counter() - t_seg
        finally:
            del loader
del model, opt, params
torch.cuda.empty_cache()

# ----- report -----
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
cols = ("heldout", "0 to 1", "1 to 3", "3 to 10", "seen_common", "seen_rare", "unseen", "d_pred_shuffle", "opt_steps", "hours")
rows = []
for ep in sorted(curve["epochs"]):
    info = curve["epochs"][ep]
    Sw = info["sampler_windows"]
    n_s = sum(w["sampled_targets"] for w in Sw)
    rows.append({"arm": "W", "epoch": ep, **{k: info["row"][k] for k in cols}, "P_med": float(np.median([w["P_med"] for w in Sw])), "P_min": float(min(w["P_min"] for w in Sw)),
                 "usable_frac": float(np.mean([w["usable_frac"] for w in Sw])),
                 "zero_frac": float(sum(w["zero_frac"] * w["sampled_targets"] for w in Sw) / max(n_s, 1)), "u_med": float(np.median([w["u_med"] for w in Sw])),
                 "sampled_targets": n_s, "grad_max_windows": info["grad_max_windows"], "upd_med_windows": info["upd_med_windows"], "peak_gib": info["peak_gib"]})
for ep in (1, 2):
    r0 = w0["epochs"]["D4"][ep]
    rows.append({"arm": "W0 (MC3ae D4)", "epoch": ep, **{k: r0["row"][k] for k in cols}, "P_med": np.nan, "P_min": np.nan, "usable_frac": np.nan, "zero_frac": np.nan,
                 "u_med": np.nan, "sampled_targets": 0, "grad_max_windows": r0["grad_max_windows"], "upd_med_windows": r0["upd_med_windows"], "peak_gib": r0["peak_gib"]})
print("\nMC3ag W UNDER D4 BY EPOCH (W0 rows from MC3ae for reference; zero_frac and u_med are sampled statistics)")
print(pd.DataFrame(rows).to_string(index=False))

def verdict(lo, hi):
    return "W better" if hi < 0 else ("W0 better" if lo > 0 else "no difference established")

if 2 in curve["epochs"]:
    d2 = curve["epochs"][2]["losses"] - w0["epochs"]["D4"][2]["losses"]
    m_, lo, hi = cluster_ci(d2, fid)
    print("\nPRIMARY ENDPOINT: W epoch 2 minus W0 epoch 2, pooled mean Huber, paired by filing, 95% interval")
    print(pd.DataFrame([{"W": float(curve["epochs"][2]["losses"].mean()), "W0": float(w0["epochs"]["D4"][2]["losses"].mean()),
                         "d_mean": m_, "ci_lo": lo, "ci_hi": hi, "verdict": verdict(lo, hi)}]).to_string(index=False))
comp = []
for ep in sorted(curve["epochs"]):
    d = curve["epochs"][ep]["losses"] - w0["epochs"]["D4"][ep]["losses"]
    for name, mk in list(bins.items()) + [(k, m) for k, m in bands.items() if m.any()] + list(groups.items()):
        m_, lo, hi = cluster_ci(d[mk], fid[mk])
        comp.append({"epoch": ep, "group": name, "targets": int(mk.sum()), "W": float(curve["epochs"][ep]["losses"][mk].mean()) if mk.any() else np.nan,
                     "W0": float(w0["epochs"]["D4"][ep]["losses"][mk].mean()) if mk.any() else np.nan, "d_mean": m_, "ci_lo": lo, "ci_hi": hi,
                     "reading": verdict(lo, hi) if np.isfinite(hi) else "not estimable"})
print("\nTRADEOFFS (reported, not gating; epoch 1 rows are descriptive): W minus W0 by fixed bin, magnitude band and concept group")
print(pd.DataFrame(comp).to_string(index=False))
pd.reset_option("display.float_format")

MC3ag contract e9588ca98bddc1ea | compares to W0 contract f1f3d18f90b0b97f (recomputed, match) | D5 facts 1d6ee181e9fd1807 | contexts bb82e1d833c2983b
initialization 88071b7a2e52af9f asserted | epoch plans c9ec2fc1ce2e0860, 8aef1402fd7de8d8 asserted | W0 steps 32,995 and 32,958
epoch: 126,767 microbatches, 1,234,902 targets | D4: step at >= 32 targets, warmup 123,490 targets, cosine to 9,879,216 | W: requested M 2048, run_factor 1, walk jump | per target usable counts sampled every 64 microbatches
resumed from checkpoint: epoch index 0, microbatch 12,680, steps 3,334 (curve taken from the checkpoint)
epoch 1 20%: steps 6,643 | heldout 0.2108 | 0 to 1 0.0694 | seen_common 0.2051 | d_pred_shuffle 0.02683 | P_med 1980 | P_min 7 | usable_frac 0.500 | zero_frac 0.0511 | u_med 989 | u_min 0 | lr 3.00e-04 | 0.65 h
epoch 1 30%: steps 10,038 | heldout 0.2021 | 0 to 1 0.0658 | seen_common 0.1959 | d_pred_shuffle 0.03606 | P_med 1980 | P_min 3 | usable_frac 0.499 | zero_frac 0.0530 | u_med 1000 |

## MC3ag findings. Subset supervision improves the cleaned reconstruction task (primary endpoint met)

Contract `e9588ca98bddc1ea` against W0 `f1f3d18f90b0b97f`. Same initialization (`88071b7a2e52af9f`), same epoch plans, same 65,953 optimizer steps, same D4 accumulation and exposure schedule, same full context evaluation. One seed, two epochs. Actual cost: W 6.24 h against W0 0.73 h, about 8.6 times the training time for the same number of updates; the comparison is at equal updates and equal data exposure, not equal compute.

**Primary endpoint (registered before the run).** W epoch 2 minus W0 epoch 2, pooled mean Huber, paired by filing, 95% interval: −0.00728 [−0.00842, −0.00628]. W 0.1592, W0 0.1665, a 4.4% lower loss. Verdict: W better. References: concept median 0.2242, zero 0.2346. The interval measures held out filing sampling uncertainty for these two trained models; it does not include training seed variability and does not undo the repeated development use of this holdout.

| Epoch 2 | W | W0 | W minus W0 | nominal interval |
|---|---|---|---|---|
| overall | 0.1592 | 0.1665 | −0.0073 | [−0.0084, −0.0063] |
| 0 to 1 | 0.0534 | 0.0545 | −0.0011 | [−0.0017, −0.0006] |
| 1 to 3 | 0.7414 | 0.7857 | −0.0443 | [−0.0499, −0.0388] |
| 3 to 10 | 2.484 | 2.600 | −0.116 | [−0.160, −0.077] |
| seen_common | 0.1509 | 0.1583 | −0.0073 | [−0.0085, −0.0063] |
| seen_rare | 0.1847 | 0.1908 | −0.0062 | [−0.0092, −0.0035] |
| unseen | 0.2217 | 0.2307 | −0.0090 | [−0.0134, −0.0049] |

Subgroup intervals are nominal and descriptive.

**Where the gain sits (fixed training bins, descriptive).** At epoch 2 W is better in bins 6 to 9 (|v| above 0.67; 47,844 of 136,446 held out targets, about 35%), with the largest difference in bin 9 (−0.069 [−0.079, −0.060]). Bins 1, 2, 4 and 5 show no established difference; bins 0 and 3 read W0 better by 0.0009 and 0.0014. The broad 0 to 1 band still improves overall (−0.0011) even though some of its constituent bins regress slightly; both statements are consistent. At epoch 1 (descriptive) W was better in bins 0 to 8 and in the 0 to 1 band by −0.0093; the small target advantage did not persist into epoch 2.

**Sampler accounting.** `P` median 1,980 against the requested 2,048, minimum 3 (small filings). Half of all drawn subsets were usable (0.500 in every window). On the sampled microbatches, 5.4% of targets received no usable subset; for those targets the loss in that microbatch is the full context contribution as weighted inside `mc_target_loss`, not a full strength W0 loss. Median usable subsets per target about 1,000.

**Other readouts.** Shuffle probe 0.139 (W) against 0.129 (W0): both arms read context. Gradient maxima 3 to 6 in every window, update norms 0.06 to 0.07, peak memory 2.9 GiB against 0.5 GiB for W0.

**What this does and does not establish.**

- Does: on the cleaned task, with a baseline that reads context, adding subset supervision (the W objective, computed through the cache) lowers held out reconstruction loss at the registered endpoint, with the interval well clear of zero, and in every band and concept group.
- Does not: establish that the cache itself improves accuracy (the cache computes the subset objective; the exactness and speed results stand separately); establish the size of the effect beyond one seed and two epochs; establish that the gain persists with more training; establish equal compute superiority; say anything about return prediction.

**Register.** Pilot primary endpoint for W against W0: met. Next: T0 against T under the same protocol, with the epoch 2 endpoint fixed and the sampler unchanged; then seed replication of W against W0 before any broader accuracy claim. Open: S1 (the 5.4% zero usable and 50% usable figures are its inputs).

## MC3ah0. Cost of the T0 and T arms before budgeting the run

Times representative microbatches of each size class (single, small, medium, full) for T0 and for T, forward and backward with the D4 accumulation form (`loss × n`), and extrapolates one epoch from the class counts of the certified epoch 1 plan. No training state is kept.

**Printed: `TIMING BY SIZE CLASS`:** per arm and class, `microbatches_timed`, `s_per_microbatch` (mean), `s_per_target`, `peak_gib`.

**Printed: `EPOCH ESTIMATE`:** per arm, `train_h` (class means times class counts), `eval_h` (21 evaluations at the MC3ae rate), `two_epochs_h`, `peak_gib`.

**Printed: `T SAMPLER (timed microbatches)`:** `P_med`, `P_min`, `usable_frac` from `batch_loss` returns on the timed T microbatches.

In [8]:
# MC3ah0. Cost of T0 and T: time representative microbatches per size class and extrapolate an epoch.
import os, sys, time
os.environ["ENV"] = "prod"
for p_ in (r"E:\apriori-ml", r"C:\Users\jakem\Documents\Belk\Sec\Bloomberg Pipeline v1\BB_Pipeline_v15"):
    if p_ not in sys.path:
        sys.path.insert(0, p_)
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import importlib
import masked_concept_model as mcm
import mc_pilot_data as mpd
import mc_batched as mcb
importlib.reload(mcm); importlib.reload(mpd); importlib.reload(mcb)
from apriori.serialization import load_from_pickle

OUT_DIR = r"E:\apriori-ml\mc3"
PILOT, CTXF = os.path.join(OUT_DIR, "mc3_pilot_data_d5.pkl.lz4"), os.path.join(OUT_DIR, "mc3_contexts_d5.pkl.lz4")
M_W, RUN_FACTOR, N_T, BATCH, MASK_RATE, ORDER_SEED_TAG, SEED = 2048, 1, 12, 32, 0.15, 131, 367
PER_CLASS, WARMUP_BATCHES, EVAL_H_EACH, N_EVALS = 12, 3, 0.02, 21
CLASSES = {"single": (1, 1), "small": (2, 8), "medium": (9, 31), "full": (32, 32)}
dev, f32 = "cuda", torch.float32

pdat = load_from_pickle(PILOT); cdat = load_from_pickle(CTXF)
indptr, indices, data, split = pdat["indptr"], pdat["indices"], pdat["data"], np.asarray(pdat["split"])
NT = torch.from_numpy(pdat["name_table"].astype(np.float32)).to(dev)
len_a = np.diff(indptr)
fact_row = np.repeat(np.arange(len(split)), len_a)
fact_local = np.arange(len(indices)) - indptr[fact_row]
tr_fact = np.isin(fact_row, np.flatnonzero(split == "train"))
occ_filing, occ_local, occ_concept = fact_row[tr_fact], fact_local[tr_fact], indices[tr_fact]
plan = mpd.column_epoch_plan(occ_concept, 0, SEED, BATCH)
order = np.random.default_rng((SEED, 0, ORDER_SEED_TAG)).permutation(len(plan))
keys = [[(int(occ_filing[o]), 0, 1, (int(occ_local[o]),)) for o in occ] for _, occ in [plan[i] for i in order]]
sizes = np.array([len(k) for k in keys])
rng = np.random.default_rng((SEED, 211))
picked = {}
for cls, (lo, hi) in CLASSES.items():
    idx = np.flatnonzero((sizes >= lo) & (sizes <= hi))
    picked[cls] = sorted(rng.choice(idx, min(PER_CLASS, len(idx)), replace=False).tolist())
counts = {cls: int(((sizes >= lo) & (sizes <= hi)).sum()) for cls, (lo, hi) in CLASSES.items()}

rows, samp = [], {"P": [], "U": 0, "Psum": 0}
for kind in ("T0", "T"):
    ds = mpd.PilotDataset(indptr, indices, data, pdat["file_names"], cdat["ctx"][N_T], kind, M=M_W, run_factor=RUN_FACTOR, seed=SEED, rate=MASK_RATE)
    torch.manual_seed(SEED); model = mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32); model.train()
    for cls, ks in picked.items():
        loader = DataLoader(ds, batch_sampler=[keys[i] for i in ks], collate_fn=mpd.collate_list, num_workers=4)
        batches = list(loader)
        for items in batches[:WARMUP_BATCHES]:                       # warm up kernels; not timed
            loss = mcb.batch_loss(model, kind, items, NT, dev, walk="levels")[0]; (loss * len(items)).backward(); model.zero_grad(set_to_none=True)
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
        t0 = time.perf_counter(); n_t = 0
        for items in batches:
            loss, exposures, _, _ = mcb.batch_loss(model, kind, items, NT, dev, walk="levels")
            (loss * len(items)).backward(); model.zero_grad(set_to_none=True)
            n_t += len(items)
            if kind == "T":
                ps = [int(it["P"]) if "P" in it else int(np.sum([len(t["terminals"]) for t in it["tries"]])) for it in items]
                samp["P"].extend(ps); samp["Psum"] += sum(ps); samp["U"] += int(exposures) - len(items)
        torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        rows.append({"arm": kind, "class": cls, "microbatches_timed": len(batches), "s_per_microbatch": dt / len(batches), "s_per_target": dt / n_t,
                     "peak_gib": torch.cuda.max_memory_allocated() / 2**30})
        del loader, batches
    del model, ds
    torch.cuda.empty_cache()

pd.set_option("display.float_format", lambda v: f"{v:.4g}")
df = pd.DataFrame(rows)
print("TIMING BY SIZE CLASS (forward and backward, D4 form)")
print(df.to_string(index=False))
est = []
for kind in ("T0", "T"):
    d = df[df["arm"] == kind].set_index("class")
    train_h = sum(d.loc[cls, "s_per_microbatch"] * counts[cls] for cls in CLASSES) / 3600
    est.append({"arm": kind, "train_h": train_h, "eval_h": EVAL_H_EACH * N_EVALS / 2, "two_epochs_h": 2 * train_h + EVAL_H_EACH * N_EVALS, "peak_gib": d["peak_gib"].max()})
print("\nEPOCH ESTIMATE (class means times class counts: " + ", ".join(f"{c} {n:,}" for c, n in counts.items()) + ")")
print(pd.DataFrame(est).to_string(index=False))
P = np.array(samp["P"])
print("\nT SAMPLER (timed microbatches)")
print(pd.DataFrame([{"P_med": float(np.median(P)), "P_min": int(P.min()), "P_max": int(P.max()), "usable_frac": samp["U"] / max(samp["Psum"], 1), "targets": len(P)}]).to_string(index=False))
pd.reset_option("display.float_format")

TIMING BY SIZE CLASS (forward and backward, D4 form)
arm  class  microbatches_timed  s_per_microbatch  s_per_target  peak_gib
 T0 single                  12          0.009512      0.009512    0.2441
 T0  small                  12           0.01589      0.005609    0.2659
 T0 medium                  12           0.01369     0.0008511     0.339
 T0   full                  12           0.01652     0.0005164    0.4294
  T single                  12           0.03553       0.03553    0.2933
  T  small                  12           0.03735       0.01318     0.521
  T medium                  12            0.0669      0.004159     1.633
  T   full                  12            0.1072      0.003351     2.074

EPOCH ESTIMATE (class means times class counts: single 61,105, small 29,405, medium 4,794, full 31,463)
arm  train_h  eval_h  two_epochs_h  peak_gib
 T0   0.4539    0.21         1.328    0.4294
  T    1.934    0.21         4.289     2.074

T SAMPLER (timed microbatches)
 P_med  P_min  P_m

## MC3ah. T0 against T on cleaned data under D4 (primary endpoint: epoch 2 versus epoch 2)

**What runs.** Two arms in sequence, T0 first. T0 attends to the target's 12 ranked context facts (full attention on those 12, no subset term); T adds the subset term over all 2^k − 1 nonempty subsets of those k ≤ 12 facts with equal weight per size class, computed through the cache on the certified level walk; the enumeration count is asserted against each target's actual context length on every microbatch. Both: same initialization as every pilot arm (`88071b7a2e52af9f`, asserted), same D3 interleaved epoch plans (hashes asserted), same D4 accumulation (step counts asserted) and exposure schedule, two epochs. **Evaluation matches the registered T design:** each held out target is predicted from its saved 12 ranked context facts, target excluded; the shuffle probe permutes the filing's values and predicts on the same ranked positions. Compared target by target, T epoch 2 against T0 epoch 2. Checkpoint authoritative, curve carried inside it.

**Printed: header:** contract, assertions passed, costs from MC3ah0, and the measured seconds of the ranked context evaluation at initialization.

**Printed at each 10% boundary and epoch end, per arm:** `arm`, `epoch`, `pct`, `opt_steps`, `heldout`, `0 to 1`, `seen_common`, `d_pred_shuffle`, `usable_frac` (T only; exact, usable subsets over enumerated subsets), `lr`, `hours`. Descriptive.

**Printed at the end**

- `MC3ah T0 AND T BY EPOCH`: `heldout`, bands, groups, `d_pred_shuffle`, `opt_steps`, `hours`, `usable_frac`, `grad_max_windows`, `upd_med_windows`, `peak_gib`.
- `PRIMARY ENDPOINT`: T epoch 2 minus T0 epoch 2, pooled mean Huber on the ranked context evaluation, paired by filing, 95% interval (held out filing sampling uncertainty for these trained models); `verdict` as in MC3ag.
- `TRADEOFFS (reported, not gating)`: by fixed bin, magnitude band and concept group; epoch 1 rows descriptive; intervals nominal.

Losses here are on the ranked context evaluation and are not comparable to the W0 and W full context losses. Files under `E:\apriori-ml\mc3`: `mc3ah_{T0,T}_last.pt`, `mc3ah_{T0,T}_epoch{1,2}.pt`, `mc3ah_curve.pkl.lz4`. Expected about 5.6 h.

In [9]:
# MC3ah. T0 then T under D4 on D5 data; ranked 12 context evaluation; primary endpoint T epoch 2 vs T0 epoch 2. Checkpoint authoritative.
import os, sys, time, math, hashlib, json, inspect
from dataclasses import asdict
os.environ["ENV"] = "prod"
for p_ in (r"E:\apriori-ml", r"C:\Users\jakem\Documents\Belk\Sec\Bloomberg Pipeline v1\BB_Pipeline_v15"):
    if p_ not in sys.path:
        sys.path.insert(0, p_)
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import importlib
import set_attention_cache, set_attention_model
import masked_concept_model as mcm
import mc_pilot_data as mpd
import mc_batched as mcb
importlib.reload(mcm); importlib.reload(mpd); importlib.reload(mcb)
from set_attention_model import signed_log
from apriori.serialization import load_from_pickle, save_to_pickle

HARNESS_VERSION = "mc3ah-2"
OUT_DIR = r"E:\apriori-ml\mc3"
PILOT, CTXF = os.path.join(OUT_DIR, "mc3_pilot_data_d5.pkl.lz4"), os.path.join(OUT_DIR, "mc3_contexts_d5.pkl.lz4")
EDGES, W0_CURVE = os.path.join(OUT_DIR, "mc3_decile_edges.pkl.lz4"), os.path.join(OUT_DIR, "mc3ae_curve.pkl.lz4")
CURVE, CURVE_TMP = os.path.join(OUT_DIR, "mc3ah_curve.pkl.lz4"), os.path.join(OUT_DIR, "mc3ah_curve_tmp.pkl.lz4")
FACTS_FP, W0_CONTRACT = "1d6ee181e9fd1807", "f1f3d18f90b0b97f"
INIT_SHA, PLAN_SHA, STEPS_AT_EPOCH = "88071b7a2e52af9f", {0: "c9ec2fc1ce2e0860", 1: "8aef1402fd7de8d8"}, {1: 32995, 2: 65953}
ARMS, WALK, EPOCHS_RUN, EPOCHS_REGISTERED = ["T0", "T"], "levels", 2, 8
M_W, RUN_FACTOR, N_T, BATCH, MASK_RATE, ORDER_SEED_TAG = 2048, 1, 12, 32, 0.15, 131
LR, WD, WARMUP_FRAC, MIN_TARGETS = 3e-4, 0.01, 0.10, 32
WORKERS, SEED, EVAL_CHUNK, N_PROBE, LOADER_TIMEOUT, MAX_STALLS, BOOT = 16, 367, 1024, 20, 300, 20, 2000
BANDS = [(0, 1, "0 to 1"), (1, 3, "1 to 3"), (3, 10, "3 to 10")]
dev, f32 = "cuda", torch.float32

def sha16(b):
    return hashlib.sha256(b).hexdigest()[:16]

def huber(pred, y):
    r = np.abs(pred - y)
    return np.where(r <= 1.0, 0.5 * r * r, r - 0.5)

def enumerated_subsets(it):
    return int(sum(len(t["terminals"]) for t in it["tries"])) if "tries" in it else int(it["P"]) * len(it["tgt"])

def expected_subsets(it):
    return int(sum(2 ** len(np.asarray(c)) - 1 for c in it["ctx"]))

# ----- data (D5), references from W0 -----
pdat = load_from_pickle(PILOT)
assert pdat["fingerprints"]["facts"] == FACTS_FP and "D5" in pdat.get("deviation", {})
cdat = load_from_pickle(CTXF)
assert cdat["pilot_facts_fp"] == FACTS_FP
w0 = load_from_pickle(W0_CURVE)
assert w0["contract"] == W0_CONTRACT
edges = load_from_pickle(EDGES)
assert edges["facts_fp"] == FACTS_FP
indptr, indices, data = pdat["indptr"], pdat["indices"], pdat["data"]
split = np.asarray(pdat["split"])
tr_idx, ho_idx = np.flatnonzero(split == "train"), np.flatnonzero(split == "holdout")
len_a = np.diff(indptr)
NT = torch.from_numpy(pdat["name_table"].astype(np.float32)).to(dev)
ctx12 = np.asarray(cdat["ctx"][N_T])                      # [n_facts, 12] local positions, -1 padded
assert ctx12.shape == (len(indices), N_T)
doc_freq = cdat["doc_freq_train"]
sl = signed_log(torch.as_tensor(np.asarray(data, dtype=np.float32))).numpy().astype(np.float64)
fact_row = np.repeat(np.arange(len(split)), len_a)
fact_local = np.arange(len(indices)) - indptr[fact_row]
tr_fact = np.isin(fact_row, tr_idx)
occ_filing, occ_local, occ_concept = fact_row[tr_fact], fact_local[tr_fact], indices[tr_fact]
N_OCC = len(occ_filing)
sources = {m.__name__: sha16(inspect.getsource(m).encode()) for m in (set_attention_cache, set_attention_model, mcm, mpd, mcb)}
split_sha = sha16(split.astype("U").tobytes())
w0_recomputed = sha16(json.dumps({"sources": sources, "facts_fp": FACTS_FP, "contexts_fp": cdat["fingerprint"], "split": split_sha,
                                  "mc_config": asdict(mcm.MCConfig()), "params": w0["params"], "harness_version": "mc3ae-1"}, sort_keys=True, default=str).encode())
assert w0_recomputed == W0_CONTRACT, f"W0 contract recomputed as {w0_recomputed}; modules, config or data differ from MC3ae"

def epoch_keys(e):
    plan = mpd.column_epoch_plan(occ_concept, e, SEED, BATCH)
    order = np.random.default_rng((SEED, e, ORDER_SEED_TAG)).permutation(len(plan))
    plan = [plan[i] for i in order]
    allocc = np.concatenate([occ for _, occ in plan])
    assert len(allocc) == N_OCC and np.array_equal(np.sort(allocc), np.arange(N_OCC)), "plan coverage"
    keys = [[(int(occ_filing[o]), e, 1, (int(occ_local[o]),)) for o in occ] for _, occ in plan]
    h = sha16(np.concatenate([np.array([k[0] for k in mb] + [k[3][0] for k in mb]) for mb in keys]).astype(np.int64).tobytes())
    assert h == PLAN_SHA[e], f"epoch {e + 1} plan hash {h} differs from the certified {PLAN_SHA[e]}"
    return keys

UPDATES_PER_EPOCH = len(epoch_keys(0))
BOUNDS = [int(round(UPDATES_PER_EPOCH * k / 10)) for k in range(1, 11)]
WARM_T, TOTAL_T = WARMUP_FRAC * N_OCC, N_OCC * EPOCHS_REGISTERED

def lr_d4(t):
    return LR * t / WARM_T if t < WARM_T else LR * 0.5 * (1 + math.cos(math.pi * min(t - WARM_T, TOTAL_T - WARM_T) / max(TOTAL_T - WARM_T, 1)))

# ----- held out targets, identical to W0; ranked context membership per target -----
ho_rows = np.concatenate([np.arange(int(indptr[i]), int(indptr[i + 1])) for i in ho_idx])
y_ho, c_ho, fid = sl[ho_rows], indices[ho_rows].astype(np.int64), fact_row[ho_rows]
assert np.array_equal(y_ho, w0["y_ho"]) and np.array_equal(c_ho, w0["c_ho"]) and np.array_equal(fid, w0["fid"]), "held out set differs from W0"
assert (ctx12[ho_rows] >= 0).any(axis=1).all(), "a held out target has an empty ranked context"
ay = np.abs(y_ho)
bands = {nm: (ay >= lo) & (ay < hi) for lo, hi, nm in BANDS}
df_ho = doc_freq[c_ho]
groups = {"seen_common": df_ho >= 50, "seen_rare": (df_ho >= 1) & (df_ho < 50), "unseen": df_ho == 0}
ho_bin = np.searchsorted(edges["assign_edges_signed_log"], ay, side="left") - 1
bins = {f"bin {b}": ho_bin == b for b in range(int(edges["n_bins"]))}

def filing_tensors(i):
    lo, hi = int(indptr[i]), int(indptr[i + 1])
    nv = NT[torch.as_tensor(indices[lo:hi].astype(np.int64), device=dev)]
    raw = torch.as_tensor(np.asarray(data[lo:hi], dtype=np.float32), device=dev)
    ctx = torch.as_tensor(ctx12[lo:hi].astype(np.int64), device=dev)                  # [n, 12]
    return nv, raw, ctx

probe_f = np.sort(np.random.default_rng((SEED, 181)).choice(ho_idx[len_a[ho_idx] <= 1024], N_PROBE, replace=False))
probe = []
for i in probe_f:
    nv, raw, ctx = filing_tensors(i)
    perm = torch.as_tensor(np.random.default_rng((SEED, 171, int(i))).permutation(nv.shape[0]), device=dev)
    probe.append((nv, raw, raw[perm], ctx))

PARAMS = {"arms": ARMS, "walk": WALK, "data": "D5", "order": "interleaved", "order_seed_tag": ORDER_SEED_TAG, "M_W": M_W, "run_factor": RUN_FACTOR, "N_T": N_T,
          "batch": BATCH, "epochs_registered": EPOCHS_REGISTERED, "epochs_run": EPOCHS_RUN, "lr": LR, "wd": WD,
          "D4": {"min_targets_per_step": MIN_TARGETS, "warmup_targets": WARM_T, "total_targets": TOTAL_T, "lr_by": "target exposure, group end"},
          "seed": SEED, "init_state_sha16": INIT_SHA, "plan_sha16": PLAN_SHA,
          "eval": "cleaned heldout, all facts, each target predicted from its 12 ranked context facts, target excluded",
          "primary_endpoint": "T epoch 2 vs T0 epoch 2, pooled mean Huber on ranked context evaluation, paired by filing"}
CONTRACT = sha16(json.dumps({"sources": sources, "facts_fp": FACTS_FP, "contexts_fp": cdat["fingerprint"], "split": split_sha,
                             "mc_config": asdict(mcm.MCConfig()), "params": PARAMS, "harness_version": HARNESS_VERSION}, sort_keys=True, default=str).encode())

def new_model():
    torch.manual_seed(SEED)
    m = mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32)
    h = sha16(b"".join(v.detach().cpu().numpy().tobytes() for v in m.state_dict().values()))
    assert h == INIT_SHA, f"initial state hash {h} differs from the certified {INIT_SHA}"
    return m

@torch.no_grad()
def predict_filing(model, nv, vals, ctx):
    """Each target attends to its ranked context positions only (ctx row, -1 padded); the target itself is never a member."""
    n = nv.shape[0]
    preds = []
    for s in range(0, n, EVAL_CHUNK):
        tg = torch.arange(s, min(n, s + EVAL_CHUNK), device=dev)
        c = torch.arange(len(tg), device=dev)
        z, V, e = model.scores(nv, vals, nv[tg])
        member = torch.zeros(len(tg), n, dtype=torch.bool, device=dev)
        cr = ctx[tg]                                                                   # [chunk, 12]
        valid = cr >= 0
        member[c[:, None].expand_as(cr)[valid], cr[valid]] = True
        assert not bool(member[c, tg].any()), "a target's ranked context contains the target"
        fp, fL, fk, fu = mcm.mc_masked_attention(z, V, member, c)
        assert bool(fu.all()), "empty ranked context at evaluation"
        preds.append(model.predict(fp, fL, fk, e))
    return torch.cat(preds).double().cpu().numpy()

@torch.no_grad()
def evaluate(model):
    t0 = time.perf_counter()
    rc, rg = torch.get_rng_state(), torch.cuda.get_rng_state_all()
    model.eval()
    out = []
    for i in ho_idx:
        nv, raw, ctx = filing_tensors(i)
        out.append(predict_filing(model, nv, raw, ctx))
    pred = np.concatenate(out)
    assert np.isfinite(pred).all()
    dp, pb = [], []
    for nv, raw, shuf, ctx in probe:
        p0, p1 = predict_filing(model, nv, raw, ctx), predict_filing(model, nv, shuf, ctx)
        dp.append(np.abs(p1 - p0)); pb.append(p0)
    model.train()
    torch.set_rng_state(rc); torch.cuda.set_rng_state_all(rg)
    l = huber(pred, y_ho)
    ev = {"heldout": float(l.mean()), **{k: float(l[m].mean()) if m.any() else float("nan") for k, m in bands.items()},
          **{k: float(l[m].mean()) for k, m in groups.items()},
          "d_pred_shuffle": float(np.median(np.concatenate(dp))), "pred_spread": float(np.std(np.concatenate(pb)))}
    return l, ev, time.perf_counter() - t0

def save_state(path, model, opt, st, curve):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "st": st, "curve": curve, "contract": CONTRACT,
                "rng_cpu": torch.get_rng_state(), "rng_cuda": torch.cuda.get_rng_state_all()}, path + ".tmp")
    os.replace(path + ".tmp", path)
    save_to_pickle(curve, CURVE_TMP, lz4_compression=True)
    os.replace(CURVE_TMP, CURVE)

def cluster_ci(d, f):
    if len(d) == 0:
        return float("nan"), float("nan"), float("nan")
    uniq, inv = np.unique(f, return_inverse=True)
    s, c = np.bincount(inv, weights=d), np.bincount(inv)
    idx = np.random.default_rng((SEED, 91)).integers(0, len(uniq), size=(BOOT, len(uniq)))
    est = s[idx].sum(1) / c[idx].sum(1)
    return float(d.mean()), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

print(f"MC3ah contract {CONTRACT} | D5 facts {FACTS_FP} | contexts {cdat['fingerprint']} | W0 contract recomputed, match")
print(f"initialization {INIT_SHA} asserted | epoch plans {PLAN_SHA[0]}, {PLAN_SHA[1]} asserted | steps at epoch end {STEPS_AT_EPOCH[1]:,} and {STEPS_AT_EPOCH[2]:,} asserted")
print(f"epoch: {UPDATES_PER_EPOCH:,} microbatches, {N_OCC:,} targets | D4: step at >= {MIN_TARGETS} targets, warmup {WARM_T:,.0f} targets, cosine to {TOTAL_T:,} | "
      f"T: all subsets of the top {N_T} ranked context facts, level walk | evaluation on ranked context | MC3ah0 training estimate: T0 0.9 h, T 3.9 h")

curve = None
for arm in ARMS:
    last = os.path.join(OUT_DIR, f"mc3ah_{arm}_last.pt")
    ds = mpd.PilotDataset(indptr, indices, data, pdat["file_names"], cdat["ctx"][N_T], arm, M=M_W, run_factor=RUN_FACTOR, seed=SEED, rate=MASK_RATE)
    model = new_model()
    params = list(model.parameters())
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)
        assert ck["contract"] == CONTRACT, f"{last} was written under contract {ck['contract']}; delete mc3ah_*.pt and {CURVE} to start over"
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); st, curve = ck["st"], ck["curve"]
        torch.set_rng_state(ck["rng_cpu"]); torch.cuda.set_rng_state_all(ck["rng_cuda"])
        print(f"[{arm}] resumed from checkpoint: epoch index {st['e']}, microbatch {st['k']:,}, steps {st['steps']:,}")
    else:
        if curve is None:
            curve = load_from_pickle(CURVE) if os.path.exists(CURVE) else {"contract": CONTRACT, "params": PARAMS, "y_ho": y_ho, "c_ho": c_ho, "fid": fid, "rows": [],
                                                                            "epochs": {a: {} for a in ARMS}}
            assert curve["contract"] == CONTRACT, f"{CURVE} was written under a different contract; delete it and mc3ah_*.pt to start over"
        _, ev0, t_ev = evaluate(model)
        st = {"e": 0, "k": 0, "t": 0, "steps": 0, "train_s": 0.0, "eval_s": t_ev, "gwin": [], "gmax": 0.0, "uwin": [], "ups": [], "tps": [], "peak": 0.0,
              "stalls": [], "pending_ckpt": False, "U": 0, "Esum": 0, "Uwin": []}
        curve["rows"].append({"arm": arm, "epoch": 0, "pct": 0, "opt_steps": 0, "lr": 0.0, **ev0, "usable_frac": np.nan, "hours": t_ev / 3600})
        print(f"[{arm}] init: heldout {ev0['heldout']:.4f} | ranked context evaluation {t_ev:.0f} s (21 evaluations per arm: {21 * t_ev / 3600:.2f} h)")
    if all((ep in curve["epochs"][arm]) for ep in range(1, EPOCHS_RUN + 1)):
        print(f"[{arm}] complete; skipping")
        del model, opt, params, ds
        continue
    model.train()
    torch.cuda.reset_peak_memory_stats()
    acc_n = 0
    opt.zero_grad(set_to_none=True)

    def do_step(count, lr):
        for p in params:
            if p.grad is not None:
                p.grad.div_(count)
        grads = [p.grad for p in params if p.grad is not None]
        if not all(bool(torch.isfinite(q).all()) for q in grads):
            raise RuntimeError(f"[{arm}] nonfinite gradient at step {st['steps']}; stopped before the update")
        st["gmax"] = max(st["gmax"], float(torch.sqrt(sum((q.double() ** 2).sum() for q in grads))))
        for gr in opt.param_groups:
            gr["lr"] = lr
        before = [p.detach().clone() for p in params]
        opt.step()
        st["ups"].append(float(torch.sqrt(sum(((p.detach() - b).double() ** 2).sum() for p, b in zip(params, before)))))
        opt.zero_grad(set_to_none=True)
        st["steps"] += 1; st["tps"].append(count)

    for e in range(st["e"], EPOCHS_RUN):
        if (e + 1) in curve["epochs"][arm]:
            continue
        keys = epoch_keys(e)
        epoch_done = False
        while not epoch_done:
            loader = DataLoader(ds, batch_sampler=keys[st["k"]:], collate_fn=mpd.collate_list, num_workers=WORKERS, prefetch_factor=2, timeout=LOADER_TIMEOUT)
            t_seg = time.perf_counter()
            try:
                for items in loader:
                    k = st["k"]
                    assert [it["i"] for it in items] == [key[0] for key in keys[k]], "loader out of plan order"
                    n = len(items)
                    if arm == "T":
                        enum, expct = sum(enumerated_subsets(it) for it in items), sum(expected_subsets(it) for it in items)
                        assert enum == expct, f"[T] enumerated {enum} subsets, expected {expct} (2^k - 1 over context lengths) at microbatch {k}"
                    loss, exposures, _, _ = mcb.batch_loss(model, arm, items, NT, dev, walk=WALK)
                    if arm == "T":
                        st["U"] += int(exposures) - n; st["Esum"] += enum
                    if not bool(torch.isfinite(loss)):
                        raise RuntimeError(f"[{arm}] nonfinite loss at epoch {e + 1} microbatch {k}")
                    st["k"] += 1; st["t"] += n
                    end = st["k"] == UPDATES_PER_EPOCH
                    (loss * n).backward()
                    acc_n += n
                    if acc_n >= MIN_TARGETS or end:
                        do_step(acc_n, lr_d4(st["t"])); acc_n = 0
                        if st["pending_ckpt"]:
                            st["train_s"] += time.perf_counter() - t_seg; t_seg = time.perf_counter()
                            save_state(last, model, opt, st, curve); st["pending_ckpt"] = False
                    if st["k"] in BOUNDS:
                        st["train_s"] += time.perf_counter() - t_seg
                        ups = np.array(st["ups"]) if st["ups"] else np.array([np.nan])
                        st["uwin"].append(round(float(np.median(ups)), 4)); st["ups"] = []
                        st["gwin"].append(round(st["gmax"], 2)); st["gmax"] = 0.0
                        uf = st["U"] / st["Esum"] if arm == "T" and st["Esum"] else np.nan
                        st["Uwin"].append(uf); st["U"] = 0; st["Esum"] = 0
                        st["peak"] = max(st["peak"], torch.cuda.max_memory_allocated() / 2**30)
                        pct = BOUNDS.index(st["k"]) * 10 + 10
                        l_new, ev, t_ev = evaluate(model)
                        st["eval_s"] += t_ev
                        lr_now = lr_d4(st["t"])
                        row = {"arm": arm, "epoch": e + 1, "pct": pct, "opt_steps": st["steps"], "lr": lr_now, **ev, "usable_frac": uf, "hours": (st["train_s"] + st["eval_s"]) / 3600}
                        curve["rows"].append(row)
                        print(f"[{arm}] epoch {e + 1} {pct}%: steps {st['steps']:,} | heldout {ev['heldout']:.4f} | 0 to 1 {ev['0 to 1']:.4f} | seen_common {ev['seen_common']:.4f} | "
                              f"d_pred_shuffle {ev['d_pred_shuffle']:.4g}" + (f" | usable_frac {uf:.3f}" if arm == "T" else "") + f" | lr {lr_now:.2e} | {row['hours']:.2f} h")
                        if end:
                            assert st["steps"] == STEPS_AT_EPOCH[e + 1], f"[{arm}] steps {st['steps']:,} differ from {STEPS_AT_EPOCH[e + 1]:,} at epoch {e + 1}"
                            tps = np.array(st["tps"])
                            curve["epochs"][arm][e + 1] = {"losses": l_new, "row": row, "grad_max_windows": st["gwin"], "upd_med_windows": st["uwin"], "usable_windows": st["Uwin"],
                                                           "peak_gib": st["peak"], "targets_per_step": (float(np.median(tps)), float(np.percentile(tps, 90)), int(tps.max()))}
                            st.update({"e": e + 1, "k": 0, "gwin": [], "uwin": [], "tps": [], "Uwin": [], "pending_ckpt": False})
                            save_state(os.path.join(OUT_DIR, f"mc3ah_{arm}_epoch{e + 1}.pt"), model, opt, st, curve)
                            save_state(last, model, opt, st, curve)
                            epoch_done = True
                        elif acc_n == 0:
                            save_state(last, model, opt, st, curve)
                        else:
                            st["pending_ckpt"] = True
                        t_seg = time.perf_counter()
                        if epoch_done:
                            break
            except RuntimeError as ex:
                if "DataLoader timed out" not in str(ex):
                    raise
                st["stalls"].append((e, st["k"]))
                print(f"[{arm}] loader stalled at epoch {e + 1} microbatch {st['k']:,}; rebuilding (stall {len(st['stalls'])} of at most {MAX_STALLS})")
                if len(st["stalls"]) >= MAX_STALLS:
                    raise RuntimeError(f"[{arm}] {MAX_STALLS} loader stalls; stopping") from None
                st["train_s"] += time.perf_counter() - t_seg
            finally:
                del loader
    del model, opt, params, ds
    torch.cuda.empty_cache()

# ----- report -----
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
cols = ("heldout", "0 to 1", "1 to 3", "3 to 10", "seen_common", "seen_rare", "unseen", "d_pred_shuffle", "opt_steps", "hours")
rows = []
for arm in ARMS:
    for ep in sorted(curve["epochs"][arm]):
        info = curve["epochs"][arm][ep]
        rows.append({"arm": arm, "epoch": ep, **{k: info["row"][k] for k in cols}, "usable_frac": float(np.nanmean(info["usable_windows"])) if arm == "T" else np.nan,
                     "grad_max_windows": info["grad_max_windows"], "upd_med_windows": info["upd_med_windows"], "peak_gib": info["peak_gib"]})
print("\nMC3ah T0 AND T BY EPOCH (ranked context evaluation; not comparable to W0 and W full context losses)")
print(pd.DataFrame(rows).to_string(index=False))

def verdict(lo, hi):
    return "T better" if hi < 0 else ("T0 better" if lo > 0 else "no difference established")

if 2 in curve["epochs"]["T"] and 2 in curve["epochs"]["T0"]:
    d2 = curve["epochs"]["T"][2]["losses"] - curve["epochs"]["T0"][2]["losses"]
    m_, lo, hi = cluster_ci(d2, fid)
    print("\nPRIMARY ENDPOINT: T epoch 2 minus T0 epoch 2, pooled mean Huber on ranked context evaluation, paired by filing, 95% interval")
    print(pd.DataFrame([{"T": float(curve["epochs"]["T"][2]["losses"].mean()), "T0": float(curve["epochs"]["T0"][2]["losses"].mean()),
                         "d_mean": m_, "ci_lo": lo, "ci_hi": hi, "verdict": verdict(lo, hi)}]).to_string(index=False))
comp = []
for ep in (1, 2):
    if ep not in curve["epochs"]["T"] or ep not in curve["epochs"]["T0"]:
        continue
    d = curve["epochs"]["T"][ep]["losses"] - curve["epochs"]["T0"][ep]["losses"]
    for name, mk in list(bins.items()) + [(k, m) for k, m in bands.items() if m.any()] + list(groups.items()):
        m_, lo, hi = cluster_ci(d[mk], fid[mk])
        comp.append({"epoch": ep, "group": name, "targets": int(mk.sum()), "T": float(curve["epochs"]["T"][ep]["losses"][mk].mean()) if mk.any() else np.nan,
                     "T0": float(curve["epochs"]["T0"][ep]["losses"][mk].mean()) if mk.any() else np.nan, "d_mean": m_, "ci_lo": lo, "ci_hi": hi,
                     "reading": verdict(lo, hi) if np.isfinite(hi) else "not estimable"})
print("\nTRADEOFFS (reported, not gating; epoch 1 rows descriptive; intervals nominal): T minus T0 by fixed bin, magnitude band and concept group")
print(pd.DataFrame(comp).to_string(index=False))
pd.reset_option("display.float_format")

MC3ah contract 972f58d9b99de2d0 | D5 facts 1d6ee181e9fd1807 | contexts bb82e1d833c2983b | W0 contract recomputed, match
initialization 88071b7a2e52af9f asserted | epoch plans c9ec2fc1ce2e0860, 8aef1402fd7de8d8 asserted | steps at epoch end 32,995 and 65,953 asserted
epoch: 126,767 microbatches, 1,234,902 targets | D4: step at >= 32 targets, warmup 123,490 targets, cosine to 9,879,216 | T: all subsets of the top 12 ranked context facts, level walk | evaluation on ranked context | MC3ah0 training estimate: T0 0.9 h, T 3.9 h
[T0] init: heldout 0.2334 | ranked context evaluation 3 s (21 evaluations per arm: 0.01 h)
[T0] epoch 1 10%: steps 3,333 | heldout 0.2007 | 0 to 1 0.0651 | seen_common 0.1919 | d_pred_shuffle 0.06656 | lr 3.00e-04 | 0.05 h
[T0] epoch 1 20%: steps 6,643 | heldout 0.1900 | 0 to 1 0.0705 | seen_common 0.1769 | d_pred_shuffle 0.1716 | lr 3.00e-04 | 0.09 h
[T0] epoch 1 30%: steps 10,038 | heldout 0.1808 | 0 to 1 0.0650 | seen_common 0.1681 | d_pred_shuffle 0.1188 | lr 3.00

## MC3ah findings. Exhaustive subset supervision over the ranked context gives a small improvement (primary endpoint met)

Contract `972f58d9b99de2d0`. Same initialization (`88071b7a2e52af9f`), same epoch plans, same 65,953 optimizer steps, same D4 accumulation and exposure schedule for both arms; ranked context evaluation (each held out target predicted from its 12 ranked facts, target excluded). One seed, two epochs. Actual cost: T 4.06 h against T0 0.84 h, about 4.8 times the training time for the same number of updates. T's subset term is exhaustively evaluated rather than estimated by sampling: every nonempty subset of each target's context entered the objective (`usable_frac` 1.000 in every window; the 2^k − 1 count was asserted on every microbatch).

**Primary endpoint (registered before the run).** T epoch 2 minus T0 epoch 2, pooled mean Huber on the ranked context evaluation, paired by filing, 95% interval: −0.00174 [−0.00241, −0.00108]. T 0.1639, T0 0.1656, a 1.1% lower loss. Verdict: T better. The interval measures held out filing sampling uncertainty for these two trained models; it does not include training seed variability.

| Epoch 2 | T | T0 | T minus T0 | nominal interval |
|---|---|---|---|---|
| overall | 0.1639 | 0.1656 | −0.0017 | [−0.0024, −0.0011] |
| 0 to 1 | 0.0549 | 0.0560 | −0.0011 | [−0.0015, −0.0007] |
| 1 to 3 | 0.7630 | 0.7726 | −0.0096 | [−0.0135, −0.0056] |
| 3 to 10 | 2.559 | 2.536 | +0.024 | [−0.005, +0.053] |
| seen_common | 0.1485 | 0.1502 | −0.0017 | [−0.0025, −0.0009] |
| seen_rare | 0.2173 | 0.2197 | −0.0024 | [−0.0034, −0.0013] |
| unseen | 0.2639 | 0.2651 | −0.0012 | [−0.0028, +0.0003] |

Subgroup intervals are nominal and descriptive.

**Where the gain sits, and the tradeoff (fixed training bins, descriptive).** At epoch 2 T is better in bins 6 to 8 (|v| from 0.67 to 2.75; largest in bin 7, −0.0144 [−0.0163, −0.0126]); bins 5 and 9 show no established difference; bins 0 to 4, which hold 75,681 of 136,446 held out targets (55.5%), read T0 better by 0.0012 to 0.0025 with intervals excluding zero. The pooled loss and the broad 0 to 1 band favor T because the larger magnitude bins outweigh those regressions; the small target tradeoff is material and stands beside the positive endpoint. At epoch 1 (descriptive) T had the lower point estimate in nine of ten bins, with the interval excluding zero in eight; as with W, the small target advantage did not persist into epoch 2. Both arms oscillated through epoch 2 (T0 0.166 to 0.176, T 0.163 to 0.174) and both endpoints sit near their lows; the fixed endpoint rule is what the verdict uses.

**Relation to the W pair, descriptive.** W over W0: −0.0073 (4.4%) at 8.6 times the cost. T over T0: −0.0017 (1.1%) at 4.8 times the cost. The losses are numerically comparable (same targets, same loss), but the two pairs evaluate different context policies (whole filing against 12 ranked facts), use different subset terms (sampled with a Horvitz Thompson estimate against exhaustive with equal weight per size class) and different objective weightings, so a difference between them cannot isolate sampling against enumeration, context selection, or objective weighting. Both read in favor of the subset term at their registered endpoints.

**Other readouts.** Shuffle probe 0.12 (T) against 0.11 (T0): both arms read their context. Gradient maxima 2 to 16 across windows, update norms 0.03 to 0.07, peak memory 2.2 GiB (T) against 0.55 GiB (T0).

**What this does and does not establish.**

- Does: on the cleaned task with the ranked context design, adding the exhaustively evaluated subset term through the cache lowers held out reconstruction loss at the registered endpoint, with the interval clear of zero, in every band and group except 3 to 10 and unseen (no difference established), with regressions in the five smallest magnitude bins.
- Does not: establish that the cache itself improves accuracy (it computes the subset objective); establish the effect beyond one seed and two epochs; establish that the small target regressions would not grow with training; establish equal compute superiority.

**Register.** Pilot primary endpoints: W against W0 met (MC3ag), T against T0 met (MC3ah). The registered pilot is complete at one seed; both results are bounded application demonstrations, not a general accuracy claim. The confirmation stage (two more paired seeds per comparison) is a separate decision and is not required for the cache paper.

## MC3ai. Cost of the W and T objectives without the cache: chunked direct enumeration against the cache walk

For sampled microbatches of each size class, the training objective (W: sampled subsets with the Horvitz Thompson term; T: all subsets of the ranked context, equal weight per size class) is run forward and backward two ways under the D4 form (`loss × n`): through the cache (`batch_loss`, jump walk for W, level walk for T) and directly, one masked attention per subset scoped to the target's own filing (W) or ranked context (T), subset gradients accumulated in row chunks into per target detached leaves, scattered once into accumulators for the encoder outputs, then one backward through the encoder. W paths containing the target keep their membership and are excluded by flag, as the walk does.

**What this benchmark is.** A target by target chunked direct baseline against the cache's batched forest. It reports numerical equivalence, measured memory and projected forward and backward hours for these two implementations. It does not establish superiority over the fastest possible direct implementation, which could also batch targets; the paper's systems claims stay anchored to the V2 and H100 direct comparisons.

**Certification before timing.** For each class, the smallest chunk size is certified first, then every chunk size the projection uses (the full class: every size in the sweep; other classes: the smallest and `CHUNK_ROWS`). Certification asserts loss and every parameter tensor's gradient equal (per tensor, absolute `ATOL` plus relative `RTOL`) and reports the worst tensor. A chunk size that runs out of memory is recorded as OOM and never timed; a class is timed at `CHUNK_ROWS` when certified, otherwise at its smallest certified chunk. Timing runs carry no assertions or host conversions inside the chunk loop.

**Printed: `EQUIVALENCE`:** per arm, class and chunk size, `status` (ok or OOM), `rel_loss_diff`, `worst_param`, `worst_err_over_tol` (below 1 passes), `subsets`, `usable`.

**Printed: `EXECUTION COST BY CLASS`:** per arm and class, `direct_chunk` (the chunk size timed), `cache_exec_s` (`batch_loss` forward and backward; includes its serial plan build), `cache_plan_s` (that plan build alone, for information), `direct_exec_s`, `direct_prep_s` (serial CPU membership and weights from the tries), `shared_trie_s` (dataset construction of the tries, shared by both paths, measured without workers), `exec_ratio`, `cache_gib`, `direct_gib`, `enumerated` (subset rows times scope facts), `evaluated` (usable rows times scope facts).

**Printed: `CHUNK SIZE SWEEP (full class)`:** `direct_exec_s`, `direct_gib` and `status` per chunk size.

**Printed: `FORWARD AND BACKWARD HOURS PER EPOCH`:** per arm, from class means times class counts: `cache_exec_h`, `direct_exec_h` (full class at its best certified chunk, other classes at their timed chunk), `direct_best_chunk`, `direct_prep_h`, `shared_trie_h`, `exec_ratio`, `prep_plus_exec_ratio` ((direct execution plus direct serial preparation) over cache execution, which already contains its serial plan build; excludes the shared trie construction; not an end to end training ratio), `run_reference_h` (actual per epoch wall time of the W run from MC3ag and the T run from MC3ah, which include optimizer updates, evaluation and checkpoints; contextual, not matched).

In [10]:
# MC3ai. Direct chunked enumeration vs the cache walk: per tensor gradient equivalence per chunk size, execution and preparation costs, epoch projection.
import os, sys, time, math
os.environ["ENV"] = "prod"
for p_ in (r"E:\apriori-ml", r"C:\Users\jakem\Documents\Belk\Sec\Bloomberg Pipeline v1\BB_Pipeline_v15"):
    if p_ not in sys.path:
        sys.path.insert(0, p_)
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import importlib
import masked_concept_model as mcm
import mc_pilot_data as mpd
import mc_batched as mcb
importlib.reload(mcm); importlib.reload(mpd); importlib.reload(mcb)
from set_attention_model import signed_log
from apriori.serialization import load_from_pickle

OUT_DIR = r"E:\apriori-ml\mc3"
PILOT, CTXF = os.path.join(OUT_DIR, "mc3_pilot_data_d5.pkl.lz4"), os.path.join(OUT_DIR, "mc3_contexts_d5.pkl.lz4")
M_W, RUN_FACTOR, N_T, BATCH, MASK_RATE, ORDER_SEED_TAG, SEED = 2048, 1, 12, 32, 0.15, 131, 367
PER_CLASS, CHUNK_ROWS, CHUNK_SWEEP, ATOL, RTOL = 6, 1024, (256, 1024, 4096), 1e-6, 1e-4
WALKS = {"W": "jump", "T": "levels"}

def _epoch_hours(path, arm):
    """Per epoch hours of an actual run from its curve file: epoch 2 end minus epoch 1 end (training plus evaluation time)."""
    if not os.path.exists(path):
        return float("nan")
    cv = load_from_pickle(path)
    ep = cv["epochs"][arm] if arm in cv["epochs"] else cv["epochs"]
    return float(ep[2]["row"]["hours"] - ep[1]["row"]["hours"]) if 2 in ep and 1 in ep else float("nan")
    
RUN_REFERENCE_H = {"W": _epoch_hours(os.path.join(OUT_DIR, "mc3ag_curve.pkl.lz4"), "W"), "T": _epoch_hours(os.path.join(OUT_DIR, "mc3ah_curve.pkl.lz4"), "T")}

CLASSES = {"single": (1, 1), "small": (2, 8), "medium": (9, 31), "full": (32, 32)}
dev, f32 = "cuda", torch.float32

pdat = load_from_pickle(PILOT); cdat = load_from_pickle(CTXF)
indptr, indices, data, split = pdat["indptr"], pdat["indices"], pdat["data"], np.asarray(pdat["split"])
NT = torch.from_numpy(pdat["name_table"].astype(np.float32)).to(dev)
len_a = np.diff(indptr)
fact_row = np.repeat(np.arange(len(split)), len_a)
fact_local = np.arange(len(indices)) - indptr[fact_row]
tr_fact = np.isin(fact_row, np.flatnonzero(split == "train"))
occ_filing, occ_local, occ_concept = fact_row[tr_fact], fact_local[tr_fact], indices[tr_fact]
plan = mpd.column_epoch_plan(occ_concept, 0, SEED, BATCH)
order = np.random.default_rng((SEED, 0, ORDER_SEED_TAG)).permutation(len(plan))
keys = [[(int(occ_filing[o]), 0, 1, (int(occ_local[o]),)) for o in occ] for _, occ in [plan[i] for i in order]]
sizes = np.array([len(k) for k in keys])
rng = np.random.default_rng((SEED, 223))
picked = {cls: sorted(rng.choice(np.flatnonzero((sizes >= lo) & (sizes <= hi)), PER_CLASS, replace=False).tolist()) for cls, (lo, hi) in CLASSES.items()}
counts = {cls: int(((sizes >= lo) & (sizes <= hi)).sum()) for cls, (lo, hi) in CLASSES.items()}

def build_prep(kind, items):
    """Serial CPU: per target, subset membership over its own scope, usable flags and fixed loss weights matching mc_target_loss."""
    prep = []
    for b, it in enumerate(items):
        n_b = int(it["n"])
        for j, t_local in enumerate(np.asarray(it["tgt"], dtype=np.int64)):
            if kind == "W":
                tr, scope, inv = it["trie"], np.arange(n_b), np.arange(n_b)
                full = np.ones(n_b, dtype=bool); full[t_local] = False
            else:
                tr, scope = it["tries"][j], np.asarray(it["ctx"][j], dtype=np.int64)
                inv = np.full(n_b, -1, dtype=np.int64); inv[scope] = np.arange(len(scope))
                full = np.ones(len(scope), dtype=bool)
            pos, parent, term = (np.asarray(tr[k_], dtype=np.int64) for k_ in ("pos", "parent", "terminals"))
            assert (inv[pos] >= 0).all(), "tree position outside the target's scope"
            P = len(term)
            member = np.zeros((P, len(scope)), dtype=bool)
            rows, cur = np.arange(P), term.copy()
            while (cur >= 0).any():                                   # a subset is the whole chain terminal -> root; depth 0 nodes are facts
                a = cur >= 0
                member[rows[a], inv[pos[cur[a]]]] = True
                cur = np.where(a, parent[np.maximum(cur, 0)], -1)
            k = member.sum(1)
            if kind == "W":
                usable = (~member[:, inv[t_local]]) & (k > 0)        # a path containing the target is unusable; membership untouched
                w = np.where(usable, 0.5 / mcm.w_subset_denominator(n_b, int(it["P"])), 0.0)
                w_full = 0.5
            else:
                usable = k > 0
                cnt = np.bincount(k[usable], minlength=int(k.max()) + 1)
                ncls = int((cnt > 0).sum())
                w = np.where(usable, 0.5 / (max(ncls, 1) * np.maximum(cnt[k], 1)), 0.0)
                w_full = 0.5 if usable.any() else 1.0
            prep.append({"b": b, "scope": scope, "full": full, "member": member, "rows_use": np.flatnonzero(usable), "usable": usable, "w": w, "w_full": w_full})
    return prep

def direct_step(model, kind, items, prep, chunk_rows, check):
    """Direct enumeration forward and backward. Leaves the gradient of (loss x n) in the parameters; returns the mean per target loss.
    check=True runs assertions and host conversions (certification); check=False keeps everything on device (timing)."""
    ns = np.array([it["n"] for it in items], dtype=np.int64)
    offs = np.concatenate([[0], np.cumsum(ns)[:-1]])
    cols = torch.as_tensor(np.concatenate([it["cols"] for it in items]), device=dev)
    vals = torch.as_tensor(np.concatenate([it["vals"] for it in items]), device=dev).to(NT.dtype)
    nv = NT[cols]
    ch_item = np.concatenate([np.full(len(it["tgt"]), b) for b, it in enumerate(items)])
    ch_global = offs[ch_item] + np.concatenate([np.asarray(it["tgt"], dtype=np.int64) for it in items])
    C = len(ch_global)
    if check:
        assert C == len(items) == len(prep), "one target per item is assumed for the D4 form"
    tg = torch.as_tensor(ch_global, device=dev)
    z, V, e = model.scores(nv, vals, nv[tg])
    y = signed_log(vals)[tg]
    zd, Vd, ed = z.detach(), V.detach(), e.detach()
    gz, gV, ge = torch.zeros_like(zd), torch.zeros_like(Vd), torch.zeros_like(ed)
    total = torch.zeros((), dtype=zd.dtype, device=dev)
    zero1 = torch.zeros(1, dtype=torch.long, device=dev)
    for c, p in enumerate(prep):
        sc = torch.as_tensor(offs[p["b"]] + p["scope"], device=dev)
        zc = zd[sc, c][:, None, :].clone().requires_grad_(True)            # per target leaves: every backward below ends here
        Vc = Vd[sc].clone().requires_grad_(True)
        ec = ed[c][None].clone().requires_grad_(True)
        fp, fL, fk, fu = mcm.mc_masked_attention(zc, Vc, torch.as_tensor(p["full"], device=dev)[None], zero1)
        if check:
            assert bool(fu.all()), "empty full context"
        lf = p["w_full"] * F.huber_loss(model.predict(fp, fL, fk, ec), y[c][None], delta=1.0, reduction="none").sum()
        lf.backward()
        total += lf.detach()
        rows_use = p["rows_use"]
        if len(rows_use):
            mem = torch.as_tensor(p["member"], device=dev)
            w = torch.as_tensor(p["w"], device=dev, dtype=zd.dtype)
            for s in range(0, len(rows_use), chunk_rows):
                idx = torch.as_tensor(rows_use[s:s + chunk_rows], device=dev)
                sp, sL, sk, su = mcm.mc_masked_attention(zc, Vc, mem[idx], torch.zeros(len(idx), dtype=torch.long, device=dev))
                if check:
                    assert bool(su.all())
                l = F.huber_loss(model.predict(sp, sL, sk, ec.expand(len(idx), -1)), y[c].expand(len(idx)), delta=1.0, reduction="none")
                lw = (w[idx] * l).sum()
                lw.backward()
                total += lw.detach()
        gz[sc, c] += zc.grad[:, 0, :]; gV[sc] += Vc.grad; ge[c] += ec.grad[0]
    torch.autograd.backward((z, V, e), (gz, gV, ge))
    return float(total) / C

def named_grads(model):
    return [(n, p.grad.detach().double().clone() if p.grad is not None else None) for n, p in model.named_parameters()]

def cache_fb(model, kind, items):
    loss = mcb.batch_loss(model, kind, items, NT, dev, walk=WALKS[kind])[0]
    (loss * len(items)).backward()
    return float(loss)

def plan_build_s(kind, items):
    """The serial plan build inside batch_loss, timed alone (information only; it is already inside cache_exec_s)."""
    plan_fn = mcb.WALKS[WALKS[kind]][0]
    ns = np.array([it["n"] for it in items], dtype=np.int64)
    offs = np.concatenate([[0], np.cumsum(ns)[:-1]])
    t0 = time.perf_counter()
    if kind == "W":
        tries = [mcb._shift(it["trie"], offs[b]) for b, it in enumerate(items) for _ in it["tgt"]]
    else:
        tries = [mcb._shift(it["tries"][j], offs[b]) for b, it in enumerate(items) for j in range(len(it["tgt"]))]
    plan_fn(tries)
    return time.perf_counter() - t0

rows_eq, rows_t, sweep = [], [], []
for kind in ("W", "T"):
    ds = mpd.PilotDataset(indptr, indices, data, pdat["file_names"], cdat["ctx"][N_T], kind, M=M_W, run_factor=RUN_FACTOR, seed=SEED, rate=MASK_RATE)
    for cls, ks in picked.items():
        t0 = time.perf_counter()
        batches = [mpd.collate_list([ds[key] for key in keys[i]]) for i in ks]           # shared trie construction, no workers
        trie_s = (time.perf_counter() - t0) / len(batches)
        # ----- certification on the first microbatch: smallest chunk first, then every chunk the projection uses -----
        torch.manual_seed(SEED); model = mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32); model.train()
        prep0 = build_prep(kind, batches[0])
        model.zero_grad(set_to_none=True); loss_c = cache_fb(model, kind, batches[0]); g_c = named_grads(model)
        model.zero_grad(set_to_none=True)
        assert math.isfinite(loss_c), f"{kind} {cls}: nonfinite cache loss"

        def certify(chunk):
            try:
                loss_d = direct_step(model, kind, batches[0], prep0, chunk, check=True)
            except torch.cuda.OutOfMemoryError:
                model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                return "OOM", float("nan"), float("nan"), ""
            g_d = named_grads(model); model.zero_grad(set_to_none=True)
            assert math.isfinite(loss_d), f"{kind} {cls} chunk {chunk}: nonfinite direct loss"
            assert all((a is None) == (b is None) for (_, a), (_, b) in zip(g_c, g_d)), f"{kind} {cls} chunk {chunk}: different parameters received gradients"
            worst, worst_name = 0.0, ""
            for (n_, a), (_, b) in zip(g_c, g_d):
                if a is None:
                    continue
                assert torch.isfinite(a).all() and torch.isfinite(b).all(), f"{kind} {cls} chunk {chunk}: nonfinite gradient in {n_}"
                r = float(((a - b).abs() / (ATOL + RTOL * a.abs().max())).max())
                if r > worst:
                    worst, worst_name = r, n_
            ldiff = abs(loss_c - loss_d) / max(abs(loss_c), 1e-12)
            assert ldiff < RTOL and worst <= 1.0, f"{kind} {cls} chunk {chunk}: direct path differs from the cache path (loss rel {ldiff:.3g}; {worst_name} err/tol {worst:.3g}); no timing taken"
            return "ok", ldiff, worst, worst_name

        chunks_needed = sorted(set(CHUNK_SWEEP) | {CHUNK_ROWS}) if cls == "full" else sorted({min(CHUNK_SWEEP), CHUNK_ROWS})
        certified = {}
        for chunk in chunks_needed:                                                   # ascending: the smallest chunk is tried first
            status, ldiff, worst, worst_name = certify(chunk)
            certified[chunk] = status == "ok"
            rows_eq.append({"arm": kind, "class": cls, "chunk_rows": chunk, "status": status, "rel_loss_diff": ldiff, "worst_param": worst_name,
                            "worst_err_over_tol": worst, "subsets": int(sum(len(p["usable"]) for p in prep0)), "usable": int(sum(p["usable"].sum() for p in prep0))})
        assert any(certified.values()), f"{kind} {cls}: no chunk size fits on this card; the direct baseline is not measurable for this class"
        # ----- cache execution (forward and backward with prepared items) -----
        for items in batches[:2]:
            cache_fb(model, kind, items); model.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
        for items in batches:
            cache_fb(model, kind, items); model.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); cache_s = (time.perf_counter() - t0) / len(batches); cache_gib = torch.cuda.max_memory_allocated() / 2**30
        plan_s = float(np.mean([plan_build_s(kind, items) for items in batches]))
        # ----- direct preparation (serial CPU) and execution at certified chunk sizes, OOM recorded per configuration -----
        t0 = time.perf_counter()
        preps = [build_prep(kind, items) for items in batches]
        prep_s = (time.perf_counter() - t0) / len(batches)

        def time_direct(chunk):
            try:
                direct_step(model, kind, batches[0], preps[0], chunk, check=False); model.zero_grad(set_to_none=True)
                torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
                for items, pr in zip(batches, preps):
                    direct_step(model, kind, items, pr, chunk, check=False); model.zero_grad(set_to_none=True)
                torch.cuda.synchronize()
                return (time.perf_counter() - t0) / len(batches), torch.cuda.max_memory_allocated() / 2**30
            except torch.cuda.OutOfMemoryError:
                model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                return float("nan"), float("nan")

        used_chunk = CHUNK_ROWS if certified.get(CHUNK_ROWS) else min(c_ for c_, ok_ in certified.items() if ok_)
        gpu_s, direct_gib = time_direct(used_chunk)
        if cls == "full":
            for chunk in CHUNK_SWEEP:
                if not certified.get(chunk):
                    sweep.append({"arm": kind, "chunk_rows": chunk, "direct_exec_s": float("nan"), "direct_gib": float("nan"), "status": "OOM at certification"})
                    continue
                s_, g_ = (gpu_s, direct_gib) if chunk == used_chunk else time_direct(chunk)
                sweep.append({"arm": kind, "chunk_rows": chunk, "direct_exec_s": s_, "direct_gib": g_, "status": "OOM" if math.isnan(s_) else "ok"})
        rows_t.append({"arm": kind, "class": cls, "direct_chunk": used_chunk, "cache_exec_s": cache_s, "cache_plan_s": plan_s, "direct_exec_s": gpu_s, "direct_prep_s": prep_s,
                       "shared_trie_s": trie_s, "exec_ratio": gpu_s / cache_s, "cache_gib": cache_gib, "direct_gib": direct_gib,
                       "enumerated": float(np.mean([sum(p["member"].size for p in pr) for pr in preps])),
                       "evaluated": float(np.mean([sum(len(p["rows_use"]) * p["member"].shape[1] for p in pr) for pr in preps]))})
        del model, batches, preps, prep0
        torch.cuda.empty_cache()
    del ds

pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print("EQUIVALENCE (direct chunked enumeration vs cache path; asserted per parameter tensor for every chunk size used; OOM recorded, never timed)")
print(pd.DataFrame(rows_eq).to_string(index=False))
dft = pd.DataFrame(rows_t)
print("\nEXECUTION COST BY CLASS (forward and backward with prepared inputs; preparation reported separately; direct at direct_chunk)")
print(dft.to_string(index=False))
dsw = pd.DataFrame(sweep)
print("\nCHUNK SIZE SWEEP (full class)")
print(dsw.to_string(index=False))
est = []
for kind in ("W", "T"):
    d = dft[dft["arm"] == kind].set_index("class")
    ok = dsw[(dsw["arm"] == kind) & (dsw["status"] == "ok")].sort_values("direct_exec_s")
    best_chunk, best_s = (int(ok.iloc[0]["chunk_rows"]), float(ok.iloc[0]["direct_exec_s"])) if len(ok) else (int(d.loc["full", "direct_chunk"]), float(d.loc["full", "direct_exec_s"]))
    dex = d["direct_exec_s"].copy(); dex["full"] = best_s
    cache_h = sum(d.loc[c, "cache_exec_s"] * counts[c] for c in CLASSES) / 3600
    direct_h = sum(dex[c] * counts[c] for c in CLASSES) / 3600
    prep_h = sum(d.loc[c, "direct_prep_s"] * counts[c] for c in CLASSES) / 3600
    trie_h = sum(d.loc[c, "shared_trie_s"] * counts[c] for c in CLASSES) / 3600
    est.append({"arm": kind, "cache_exec_h": cache_h, "direct_exec_h": direct_h, "direct_best_chunk": best_chunk, "direct_prep_h": prep_h, "shared_trie_h": trie_h,
                "exec_ratio": direct_h / cache_h, "prep_plus_exec_ratio": (direct_h + prep_h) / cache_h,
                "run_reference_h": RUN_REFERENCE_H[kind] if RUN_REFERENCE_H[kind] is not None else np.nan})
print("\nFORWARD AND BACKWARD HOURS PER EPOCH (class means times class counts: " + ", ".join(f"{c} {n:,}" for c, n in counts.items())
      + "; full class at its best certified chunk, other classes at direct_chunk; no optimizer, evaluation or checkpoints)")
print("prep_plus_exec_ratio = (direct execution + direct serial preparation) / cache execution (which already contains its serial plan build); shared trie construction excluded")
print(pd.DataFrame(est).to_string(index=False))
pd.reset_option("display.float_format")

EQUIVALENCE (direct chunked enumeration vs cache path; asserted per parameter tensor for every chunk size used; OOM recorded, never timed)
arm  class  chunk_rows status  rel_loss_diff      worst_param  worst_err_over_tol  subsets  usable
  W single         256     ok      8.904e-08   query.mask_vec            0.002553     2002    1004
  W single        1024     ok              0   query.mask_vec            0.002043     2002    1004
  W  small         256     ok      4.747e-07 readout.5.weight            0.001729     5869    2614
  W  small        1024     ok      3.021e-07 readout.1.weight             0.00171     5869    2614
  W medium         256     ok      1.663e-07 readout.1.weight            0.007043    40897   22189
  W medium        1024     ok      1.109e-08    encoder.w_val            0.007413    40897   22189
  W   full         256     ok       8.46e-08 readout.3.weight            0.007381    63394   29048
  W   full        1024     ok       8.46e-08 readout.1.weight        

## MC3ai findings. Scoped direct computation is competitive at pilot scale; cache performance depends on workload and implementation

**Certification.** On the first microbatch of every class and at every chunk size used, the direct chunked path reproduced the cache path's loss to 5e-7 relative or better and every parameter tensor's gradient within tolerance (worst error 1.5% of the tolerance, `encoder.w_val`, T medium at 256 rows). Both implementations train the same objective; the comparison below is between implementations only.

**Forward and backward hours per epoch (projected from class means; no optimizer, evaluation or checkpoints)**

| Arm | cache execution | direct execution (best chunk 4,096) | ratio, execution (direct over cache) | direct serial preparation | ratio, preparation plus execution | shared preparation in this harness | actual run, per epoch |
|---|---|---|---|---|---|---|---|
| W | 2.54 h | 2.07 h | 0.81 | 1.38 h | 1.35 | 5.29 h | 3.11 h |
| T | 1.86 h | 2.39 h | 1.29 | 0.28 h | 1.44 | 0.03 h | 2.01 h |

**Measured execution.**

- W: direct takes 0.81 times the cache time; direct is faster.
- T: direct takes 1.29 times the cache time; the cache is faster.
- With direct's serial preparation added, direct is 1.35 (W) and 1.44 (T) times the cache's execution, which already contains its serial plan build. These are serial preparation plus execution comparisons, not measured training speedups; the actual runs overlap preparation with workers.
- Memory: direct peaks at 0.52 GiB against the cache's 2.65 GiB (W full) and 2.13 GiB (T full). Direct is the lighter path.
- Chunk size: 256 to 4,096 rows moved W from 0.44 s to 0.19 s per full microbatch and T from 1.30 s to 0.19 s. The direct figures use the best measured chunk for the full class and 1,024 for the other classes, so direct could be somewhat faster still.

**Shared preparation.** The 5.29 h per W epoch is serial dataset construction in this harness (sampler draws and trie building), measured without workers; the training runs overlap it across 16 workers. Direct enumeration does not need a trie; it builds subsets from the cache's representation here only so that both paths evaluate identical subsets.

**What this cell does not isolate.** Structural reuse is present in both workloads (T's complete 12 fact power set has 4,095 nodes against 24,576 summed subset facts, about sixfold; W's sampler creates overlapping prefixes by design, as the benchmark notebooks show). Hardware overhead of the level and jump walks, the extra work of pointer jumping, and the batching efficiency of the direct path can outweigh structural savings at these sizes; this cell measures the net effect for these implementations and does not apportion it.

**Defensible conclusion.** Both objectives are feasible on the 2080 with either implementation. Direct is lighter in memory; W favors direct execution, T modestly favors cache execution. These application results qualify the performance story and change neither learning endpoint nor the separate systems benchmarks.

**Limits.** Target by target chunked direct baseline against the cache's batched forest; a batched direct implementation could be faster than measured. Six sampled microbatches per class, one sampler seed. Projections are forward and backward only.

**Register.** No change to the pilot verdicts. The paper's systems claims rest on the Powerset Validation Cache Benchmarks (2080 and H100); this cell is cited for the certified equivalence of the training paths and for the measured competitiveness of scoped direct computation at pilot scale.

In [11]:
# MC3ai1. Save the MC3ai tables already in the kernel, with settings and fingerprints; no new measurement.
import inspect, hashlib, platform
import set_attention_cache, set_attention_model

def sha16(b):
    return hashlib.sha256(b).hexdigest()[:16]

OUT = os.path.join(OUT_DIR, "mc3ai_results.pkl.lz4")
record = {
    "harness": "MC3ai", "equivalence": pd.DataFrame(rows_eq), "execution_by_class": dft, "chunk_sweep": dsw, "epoch_estimate": pd.DataFrame(est),
    "settings": {"PER_CLASS": PER_CLASS, "CHUNK_ROWS": CHUNK_ROWS, "CHUNK_SWEEP": CHUNK_SWEEP, "ATOL": ATOL, "RTOL": RTOL, "WALKS": WALKS,
                 "M_W": M_W, "RUN_FACTOR": RUN_FACTOR, "N_T": N_T, "BATCH": BATCH, "MASK_RATE": MASK_RATE, "ORDER_SEED_TAG": ORDER_SEED_TAG, "SEED": SEED,
                 "picked_microbatches": picked, "class_counts": counts, "run_reference_h": RUN_REFERENCE_H},
    "fingerprints": {"facts": pdat["fingerprints"]["facts"], "contexts": cdat["fingerprint"],
                     "sources": {m.__name__: sha16(inspect.getsource(m).encode()) for m in (set_attention_cache, set_attention_model, mcm, mpd, mcb)}},
    "environment": {"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0), "python": platform.python_version()},
}
save_to_pickle(record, OUT, lz4_compression=True)
print(f"saved: {OUT} ({os.path.getsize(OUT) / 2**10:,.0f} KiB)")
print(record["fingerprints"])
print(record["environment"])

saved: E:\apriori-ml\mc3\mc3ai_results.pkl.lz4 (4 KiB)
{'facts': '1d6ee181e9fd1807', 'contexts': 'bb82e1d833c2983b', 'sources': {'set_attention_cache': 'f53957e0033d0c72', 'set_attention_model': '647b44ba68fe9dd4', 'masked_concept_model': 'e8c9321790cbddd3', 'mc_pilot_data': 'f7b8e10a079077d5', 'mc_batched': '0be0f3440e96db76'}}
{'torch': '2.5.1+cu121', 'cuda': '12.1', 'gpu': 'NVIDIA GeForce RTX 2080 SUPER', 'python': '3.12.7'}
